# Draft Weight Estimation — Role-Scaled Player Baseline with Regularized Pick Calibration

This notebook preserves the finalized efficiency × load player-production components and the regularized draft-pick calibration, while changing one structural assumption:

> A low-minute player should not receive the same automatic baseline value as a proven full-rotation player.

The base portion of player value is now scaled by demonstrated role capacity. The quality adjustment remains reliability- and role-adjusted, and multi-player packages retain the slot-cost rule.


## Model inputs and assumptions

The model uses the finalized player-production components, reliability and role adjustments, a role-scaled player baseline, a fixed relative draft-tier curve, and regularized calibration of the absolute pick-value scale.


In [1]:
# Load modeling libraries and configure processed outputs.
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import percentileofscore
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

processed_data_path = Path("../data/processed")
processed_data_path.mkdir(parents=True, exist_ok=True)

In [2]:
# Load model data, player features, season records, and configuration.
draft_weight_model_data = pd.read_parquet(processed_data_path / "draft_weight_model_data_additive.parquet")

transaction_player_data = pd.read_parquet(processed_data_path / "transaction_player_features_for_draft_value.parquet")

team_seasons = pd.read_csv("../data/interim/team_season_records.csv")

with open(processed_data_path / "draft_weight_model_config_additive.json", "r", encoding="utf-8") as file:
    draft_weight_config = json.load(file)

print("Model rows:", len(draft_weight_model_data))
print("Transaction-player rows:", len(transaction_player_data))

Model rows: 1963
Transaction-player rows: 6468


## Model definitions


In [3]:
# Restore the ordered pick hierarchy and modeling constants.
outright_pick_hierarchy = draft_weight_config["outright_pick_hierarchy"]
swap_draft_tiers = draft_weight_config["swap_draft_tiers"]

outright_pick_net_columns = [f"net_{tier}_count" for tier in outright_pick_hierarchy]

swap_net_columns = [f"net_{tier}_count" for tier in swap_draft_tiers]

MINIMUM_SEASON_MINUTES = 100.0
BASE_PLAYER_VALUE = 1.0
MINIMUM_PLAYER_BASE = 0.20
QUALITY_MULTIPLIER_SLOPE = 0.5
ZSCORE_CLIP_LIMIT = 3.0

ROLE_CAPACITY_FULL_MINUTES_PER_GAME = 30.0
ROLE_CAPACITY_EXPONENT = 0.5

FINAL_TEST_SEASON_COUNT = 5
INTERNAL_VALIDATION_SEASON_COUNT = 5

CANDIDATE_RELIABILITY_SHRINKAGE_CONSTANTS = [250.0, 500.0, 750.0, 1000.0]

CANDIDATE_ADDITIONAL_PLAYER_SLOT_COSTS = [0.00, 0.25, 0.50, 0.75, 1.00]

# Hyperparameter selection emphasizes trades where package concentration matters.
VALIDATION_SAMPLE_WEIGHTS = {
    "one_player_vs_multiple_players_with_picks": 0.40,
    "unequal_player_counts_with_picks": 0.30,
    "multi_player_packages_with_picks": 0.20,
    "all_complete_packages_with_picks": 0.10,
}

# Any candidate within one percentage point of the best normalized validation
# score is treated as statistically indistinguishable for tie-breaking.
VALIDATION_SCORE_TOLERANCE = 0.01

# Face-validity guardrails for package concentration.
MINIMUM_P95_TO_TWO_MEDIAN_PACKAGE_RATIO = 1.00
MINIMUM_SECOND_MEDIAN_CONTRIBUTION_SHARE = 0.10


# Previously accepted absolute pick values. These define the prior scale only;
# the new player model can move the deployed scale when transaction evidence
# supports a change.
PRIOR_DEPLOYED_PICK_VALUES = {
    "projected_late_second": 0.239096,
    "projected_early_second": 0.239096,
    "projected_late_first": 0.586212,
    "projected_lottery_first": 1.195620,
    "projected_top_5_first": 1.415349,
}

# The zero-strength candidate is retained as an unregularized diagnostic.
# Final model selection is restricted to positive strengths.
CANDIDATE_PICK_SCALE_REGULARIZATION_STRENGTHS = [0.0, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0]

PICK_SCALE_VALIDATION_SAMPLE_WEIGHTS = {
    "one_calculated_player_for_zero_with_picks": 0.50,
    "unequal_player_counts_with_picks": 0.30,
    "all_complete_packages_with_picks": 0.20,
}

# Prefer stronger regularization when validation scores are effectively tied.
PICK_SCALE_VALIDATION_SCORE_TOLERANCE = 0.01

PICK_SCALE_BOOTSTRAP_RESAMPLES = 500
PICK_SCALE_BOOTSTRAP_RANDOM_SEED = 20260805

# Metrics standardized after the development-period TS baseline is calculated.
player_metric_columns = [
    "season_true_shooting_attempts_per_100",
    "season_efficiency_points_added_per_100",
    "season_assists_per_100",
    "season_turnovers_per_100",
    "season_total_rebound_percentage",
    "season_steal_percentage",
    "season_block_percentage",
    "season_plus_minus_per_100",
]

# Each domain contains signed within-domain metric weights. Absolute weights
# within each domain sum to one so domain scales remain broadly comparable.
player_domain_metric_weights = {
    "scoring_load": {"season_true_shooting_attempts_per_100_zscore": 1.0},
    "scoring_efficiency": {"season_efficiency_points_added_per_100_zscore": 1.0},
    "playmaking": {"season_assists_per_100_zscore": 2.0 / 3.0, "season_turnovers_per_100_zscore": -1.0 / 3.0},
    "rebounding_defensive_events": {
        "season_total_rebound_percentage_zscore": 1.0 / 3.0,
        "season_steal_percentage_zscore": 1.0 / 3.0,
        "season_block_percentage_zscore": 1.0 / 3.0,
    },
    "overall_impact": {"season_plus_minus_per_100_zscore": 1.0},
}

player_domains = {domain_name: list(metric_weights) for domain_name, metric_weights in player_domain_metric_weights.items()}

player_domain_weights = {
    "scoring_load": 0.30,
    "scoring_efficiency": 0.20,
    "playmaking": 0.25,
    "rebounding_defensive_events": 0.10,
    "overall_impact": 0.15,
}

if set(player_domains) != set(player_domain_weights):
    raise ValueError("player domains and domain weights must contain the same domains.")

for domain_name, metric_weights in player_domain_metric_weights.items():
    absolute_weight_sum = float(sum(abs(value) for value in metric_weights.values()))
    if not np.isclose(absolute_weight_sum, 1.0):
        raise ValueError(f"Absolute within-domain weights for {domain_name} must sum to 1.0; " f"received {absolute_weight_sum:.6f}.")

domain_weight_sum = float(sum(player_domain_weights.values()))
if not np.isclose(domain_weight_sum, 1.0):
    raise ValueError("Player domain weights must sum to 1.0; " f"received {domain_weight_sum:.6f}.")

direct_metric_weights = {}
for domain_name, metric_weights in player_domain_metric_weights.items():
    for z_column, within_domain_weight in metric_weights.items():
        metric = z_column.removesuffix("_zscore")
        direct_metric_weights[metric] = float(player_domain_weights[domain_name]) * float(within_domain_weight)

direct_metric_weight_table = (
    pd.DataFrame(
        {
            "metric": list(direct_metric_weights),
            "signed_direct_weight": list(direct_metric_weights.values()),
            "absolute_direct_weight": [abs(value) for value in direct_metric_weights.values()],
        }
    )
    .sort_values("absolute_direct_weight", ascending=False)
    .reset_index(drop=True)
)

# Existing rate columns that the future current-roster builder can map directly.
current_roster_metric_mapping = {
    "season_assists_per_100": "ASSISTS_PER_100",
    "season_total_rebound_percentage": "TOTAL_REBOUND_PERCENTAGE",
    "season_steal_percentage": "STEAL_PERCENTAGE",
    "season_block_percentage": "BLOCK_PERCENTAGE",
    "season_plus_minus_per_100": "PLUS_MINUS_PER_100",
}

# Raw inputs needed to reconstruct the new shooting-load, efficiency-added,
# turnover-volume, reliability, and role-capacity metrics in the app pipeline.
current_roster_raw_column_mapping = {
    "season_games_played": "CALCULATED_GP",
    "season_minutes": "CALCULATED_MIN",
    "season_estimated_player_possessions": "ESTIMATED_PLAYER_POSSESSIONS",
    "season_field_goals_attempted": "CALCULATED_FGA",
    "season_free_throws_attempted": "CALCULATED_FTA",
    "season_turnovers": "CALCULATED_TOV",
    "season_true_shooting_percentage": "TRUE_SHOOTING_PERCENTAGE",
}

# Preserve the relative spacing from the original package-quality draft curve.
original_quality_model_weights = pd.Series(
    {
        "projected_late_second": 0.081432,
        "projected_early_second": 0.081432,
        "projected_late_first": 0.199654,
        "projected_lottery_first": 0.407208,
        "projected_top_5_first": 0.482044,
    },
    dtype="float64",
)

missing_original_tiers = [tier for tier in outright_pick_hierarchy if tier not in original_quality_model_weights.index]
if missing_original_tiers:
    raise KeyError("The original pick curve is missing tiers: " f"{missing_original_tiers}")

relative_pick_curve = (
    original_quality_model_weights.reindex(outright_pick_hierarchy) / original_quality_model_weights["projected_top_5_first"]
)


prior_pick_weight_series = pd.Series(PRIOR_DEPLOYED_PICK_VALUES, dtype="float64").reindex(outright_pick_hierarchy)

if prior_pick_weight_series.isna().any():
    missing_prior_tiers = prior_pick_weight_series.index[prior_pick_weight_series.isna()].tolist()
    raise KeyError("The prior deployed pick curve is missing tiers: " f"{missing_prior_tiers}")

PRIOR_PICK_CURVE_SCALE = float(PRIOR_DEPLOYED_PICK_VALUES["projected_top_5_first"])

prior_relative_curve = prior_pick_weight_series / PRIOR_PICK_CURVE_SCALE

if not np.allclose(prior_relative_curve.to_numpy(dtype="float64"), relative_pick_curve.to_numpy(dtype="float64"), atol=1e-4, rtol=0.0):
    raise ValueError("The prior deployed values do not follow the fixed relative curve.")

relative_pick_curve_table = pd.DataFrame(
    {
        "tier": outright_pick_hierarchy,
        "original_quality_model_weight": (original_quality_model_weights.reindex(outright_pick_hierarchy).to_numpy()),
        "relative_curve_weight": relative_pick_curve.to_numpy(),
    }
)

display(direct_metric_weight_table)
display(relative_pick_curve_table)

,metric,signed_direct_weight,absolute_direct_weight
0,season_true_shooting_attempts_per_100,0.300000,0.300000
1,season_efficiency_points_added_per_100,0.200000,0.200000
2,season_assists_per_100,0.166667,0.166667
3,season_plus_minus_per_100,0.150000,0.150000
4,season_turnovers_per_100,-0.083333,0.083333
5,season_total_rebound_percentage,0.033333,0.033333
6,season_steal_percentage,0.033333,0.033333
7,season_block_percentage,0.033333,0.033333


,tier,original_quality_model_weight,relative_curve_weight
0,projected_late_second,0.081432,0.168931
1,projected_early_second,0.081432,0.168931
2,projected_late_first,0.199654,0.414182
3,projected_lottery_first,0.407208,0.844753
4,projected_top_5_first,0.482044,1.000000


## Assign production-reference seasons


In [4]:
# Build season calendar helpers for time-based splits.
def build_season_calendar(team_seasons):
    required_columns = {"Season", "season_start_date", "season_end_date"}
    missing_columns = required_columns - set(team_seasons.columns)
    if missing_columns:
        raise KeyError(f"team_seasons is missing: {sorted(missing_columns)}")

    calendar = team_seasons[["Season", "season_start_date", "season_end_date"]].copy()

    calendar["season_start_date"] = pd.to_datetime(calendar["season_start_date"], errors="coerce")
    calendar["season_end_date"] = pd.to_datetime(calendar["season_end_date"], errors="coerce")

    calendar = (
        calendar.groupby("Season", as_index=False)
        .agg(season_start_date=("season_start_date", "min"), season_end_date=("season_end_date", "max"))
        .sort_values("season_start_date")
        .reset_index(drop=True)
    )

    if calendar[["season_start_date", "season_end_date"]].isna().any().any():
        raise ValueError("Season calendar contains missing dates.")

    return calendar


def assign_production_reference_season(data, date_column, season_calendar):
    result = data.copy()
    result[date_column] = pd.to_datetime(result[date_column], errors="coerce")

    if result[date_column].isna().any():
        raise ValueError(f"{date_column} contains invalid dates.")

    result["_original_row_order"] = np.arange(len(result))

    reference_calendar = season_calendar[["Season", "season_start_date", "season_end_date"]].rename(
        columns={"Season": "production_reference_season"}
    )

    result = pd.merge_asof(
        result.sort_values(date_column),
        reference_calendar.sort_values("season_start_date"),
        left_on=date_column,
        right_on="season_start_date",
        direction="backward",
    )

    result = result.sort_values("_original_row_order").drop(columns="_original_row_order").reset_index(drop=True)

    result["production_reference_season_start_year"] = pd.to_numeric(
        result["production_reference_season"].astype("string").str.extract(r"^(\d{4})", expand=False), errors="coerce"
    )

    if result["production_reference_season_start_year"].isna().any():
        raise ValueError("Some rows could not be assigned a reference season.")

    return result


season_calendar = build_season_calendar(team_seasons)

draft_weight_model_data = assign_production_reference_season(draft_weight_model_data, date_column="Date", season_calendar=season_calendar)

transaction_player_data = assign_production_reference_season(
    transaction_player_data, date_column="transaction_date", season_calendar=season_calendar
)

## Create parameter-training, internal-validation, and untouched test periods


In [5]:
# Assign development and test seasons without future leakage.
ordered_seasons = (
    draft_weight_model_data[["production_reference_season", "production_reference_season_start_year"]]
    .drop_duplicates()
    .sort_values("production_reference_season_start_year")["production_reference_season"]
    .tolist()
)

minimum_required_seasons = FINAL_TEST_SEASON_COUNT + INTERNAL_VALIDATION_SEASON_COUNT + 3
if len(ordered_seasons) < minimum_required_seasons:
    raise ValueError("Not enough seasons for parameter training, internal validation, " "and an untouched final test period.")

test_seasons = ordered_seasons[-FINAL_TEST_SEASON_COUNT:]
development_seasons = ordered_seasons[:-FINAL_TEST_SEASON_COUNT]

validation_season_count = min(INTERNAL_VALIDATION_SEASON_COUNT, max(2, len(development_seasons) // 4))
validation_seasons = development_seasons[-validation_season_count:]
parameter_training_seasons = development_seasons[:-validation_season_count]

if len(parameter_training_seasons) < 3:
    raise ValueError("The parameter-training period must contain at least three seasons.")

print("Parameter-training seasons:", parameter_training_seasons[0], "through", parameter_training_seasons[-1])
print("Internal-validation seasons:", validation_seasons)
print("Final untouched test seasons:", test_seasons)

Parameter-training seasons: 1985-86 through 2008-09
Internal-validation seasons: ['2009-10', '2010-11', '2011-12', '2012-13', '2013-14']
Final untouched test seasons: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19']


## Build efficiency × load, reliability-shrunk, role-adjusted individual player values

The production components remain unchanged. The revised individual value is:

\[
\text{role-scaled base}
=
b_{min} + (b_{full}-b_{min})\times \text{role capacity}
\]

\[
\text{player value}
=
\max\left(
0,
\text{role-scaled base}
+
\beta \times \text{role-adjusted quality}
\right)
\]

The default minimum base is `0.20`; a player reaches the full `1.00` baseline only at full role capacity.


In [6]:
# Convert player box-score totals into comparable rate metrics.
def prepare_player_rate_metrics(player_rows, reference_true_shooting_percentage=None):
    result = player_rows.copy()

    required_base_columns = [
        "season_games_played",
        "season_minutes",
        "season_estimated_player_possessions",
        "season_true_shooting_percentage",
        "season_assists_per_100",
        "season_total_rebound_percentage",
        "season_steal_percentage",
        "season_block_percentage",
        "season_plus_minus_per_100",
    ]

    for column in required_base_columns:
        if column not in result.columns:
            raise KeyError(f"Transaction-player data is missing: {column}")
        result[column] = pd.to_numeric(result[column], errors="coerce")

    possessions = pd.to_numeric(result["season_estimated_player_possessions"], errors="coerce")

    # Prefer raw FGA and FTA. Fall back to points and TS when those counts were
    # not included in an older transaction-player export.
    raw_fga_available = "season_field_goals_attempted" in result.columns
    raw_fta_available = "season_free_throws_attempted" in result.columns

    if raw_fga_available and raw_fta_available:
        fga = pd.to_numeric(result["season_field_goals_attempted"], errors="coerce")
        fta = pd.to_numeric(result["season_free_throws_attempted"], errors="coerce")
        true_shooting_attempts = fga + 0.44 * fta
        shooting_load_source = "raw_fga_fta"
    else:
        if "season_points" in result.columns:
            points = pd.to_numeric(result["season_points"], errors="coerce")
        elif "season_points_per_100" in result.columns:
            points = pd.to_numeric(result["season_points_per_100"], errors="coerce") * possessions / 100.0
        else:
            raise KeyError(
                "The player data must contain either season_field_goals_attempted "
                "and season_free_throws_attempted, or season_points / "
                "season_points_per_100 for the true-shooting-attempt fallback."
            )

        true_shooting = pd.to_numeric(result["season_true_shooting_percentage"], errors="coerce")
        true_shooting_attempts = np.where(true_shooting.gt(0) & points.notna(), points / (2.0 * true_shooting), np.nan)
        true_shooting_attempts = pd.Series(true_shooting_attempts, index=result.index, dtype="float64")
        shooting_load_source = "derived_from_points_and_ts"

    result["season_true_shooting_attempts"] = pd.to_numeric(true_shooting_attempts, errors="coerce")
    result["season_true_shooting_attempts_per_100"] = np.where(
        possessions.gt(0) & result["season_true_shooting_attempts"].notna(),
        100.0 * result["season_true_shooting_attempts"] / possessions,
        np.nan,
    )
    result["shooting_load_source"] = shooting_load_source

    # Prefer raw turnovers. Fall back to the exact TOV% identity used by the
    # original feature pipeline: TOV% = 100*TOV/(FGA + 0.44*FTA + TOV).
    if "season_turnovers" in result.columns:
        turnovers = pd.to_numeric(result["season_turnovers"], errors="coerce")
        turnover_volume_source = "raw_turnovers"
    elif "season_turnover_percentage" in result.columns:
        turnover_percentage = pd.to_numeric(result["season_turnover_percentage"], errors="coerce")
        turnover_rate = turnover_percentage / 100.0
        turnovers = np.where(
            turnover_rate.ge(0) & turnover_rate.lt(1) & result["season_true_shooting_attempts"].notna(),
            (turnover_rate * result["season_true_shooting_attempts"] / (1.0 - turnover_rate)),
            np.nan,
        )
        turnovers = pd.Series(turnovers, index=result.index, dtype="float64")
        turnover_volume_source = "derived_from_turnover_percentage"
    else:
        raise KeyError("The player data must contain season_turnovers or " "season_turnover_percentage.")

    result["season_turnovers_reconstructed"] = pd.to_numeric(turnovers, errors="coerce")
    result["season_turnovers_per_100"] = np.where(
        possessions.gt(0) & result["season_turnovers_reconstructed"].notna(),
        100.0 * result["season_turnovers_reconstructed"] / possessions,
        np.nan,
    )
    result["turnover_volume_source"] = turnover_volume_source

    result["season_minutes_per_game"] = np.where(
        result["season_games_played"].gt(0) & result["season_minutes"].notna(),
        result["season_minutes"] / result["season_games_played"],
        np.nan,
    )

    role_ratio = np.clip(result["season_minutes_per_game"] / float(ROLE_CAPACITY_FULL_MINUTES_PER_GAME), 0.0, 1.0)
    result["season_player_role_capacity"] = np.power(role_ratio, float(ROLE_CAPACITY_EXPONENT))

    if reference_true_shooting_percentage is None:
        result["season_efficiency_points_added_per_100"] = np.nan
    else:
        reference_ts = float(reference_true_shooting_percentage)
        result["season_efficiency_points_added_per_100"] = (
            2.0 * result["season_true_shooting_attempts_per_100"] * (result["season_true_shooting_percentage"] - reference_ts)
        )
        result["reference_true_shooting_percentage"] = reference_ts

    return result


def known_no_prior_production_mask(player_rows):
    return player_rows["player_match_status"].eq("no_prior_nba_appearance") | player_rows["feature_status"].eq(
        "no_prior_box_score_appearance"
    )


def base_reference_player_mask(player_rows):
    base_metric_columns = [
        "season_true_shooting_attempts_per_100",
        "season_true_shooting_percentage",
        "season_assists_per_100",
        "season_turnovers_per_100",
        "season_total_rebound_percentage",
        "season_steal_percentage",
        "season_block_percentage",
        "season_plus_minus_per_100",
        "season_player_role_capacity",
    ]

    return (
        player_rows["feature_status"].eq("calculated")
        & player_rows["season_games_played"].gt(0)
        & player_rows["season_minutes"].ge(MINIMUM_SEASON_MINUTES)
        & player_rows["season_estimated_player_possessions"].gt(0)
        & player_rows[base_metric_columns].notna().all(axis=1)
    )


def qualified_rate_player_mask(player_rows):
    return base_reference_player_mask(player_rows) & player_rows[player_metric_columns].notna().all(axis=1)


def below_minimum_minutes_mask(player_rows):
    return (
        player_rows["feature_status"].eq("calculated")
        & player_rows["season_minutes"].notna()
        & player_rows["season_minutes"].lt(MINIMUM_SEASON_MINUTES)
    )


def deduplicate_metric_reference_rows(reference_rows):
    result = reference_rows.copy()

    if "player_id" in result.columns:
        usable_id = result["player_id"].notna()
        with_id = result.loc[usable_id].drop_duplicates(subset=["player_id", "transaction_date"])
        without_id = result.loc[~usable_id].drop_duplicates(subset=["transaction_player_name", "transaction_date"])
        return pd.concat([with_id, without_id], ignore_index=True)

    return result.drop_duplicates(subset=["transaction_player_name", "transaction_date"]).reset_index(drop=True)


def build_player_metric_reference(reference_player_rows):
    prepared = prepare_player_rate_metrics(reference_player_rows, reference_true_shooting_percentage=None)
    valid_reference = prepared.loc[base_reference_player_mask(prepared)].copy()
    valid_reference = deduplicate_metric_reference_rows(valid_reference)

    if valid_reference.empty:
        raise ValueError("No players meet the complete 100-minute " "efficiency-load reference rule.")

    ts_weights = pd.to_numeric(valid_reference["season_true_shooting_attempts"], errors="coerce")
    ts_values = pd.to_numeric(valid_reference["season_true_shooting_percentage"], errors="coerce")
    valid_ts = ts_weights.gt(0) & ts_values.notna()

    if not valid_ts.any():
        raise ValueError("No valid shooting-attempt-weighted TS observations are available.")

    reference_true_shooting_percentage = float(np.average(ts_values.loc[valid_ts], weights=ts_weights.loc[valid_ts]))

    valid_reference = prepare_player_rate_metrics(valid_reference, reference_true_shooting_percentage=(reference_true_shooting_percentage))
    valid_reference = valid_reference.loc[qualified_rate_player_mask(valid_reference)].copy()

    parameter_table = pd.DataFrame(
        {
            "metric": player_metric_columns,
            "mean": [valid_reference[column].mean() for column in player_metric_columns],
            "std_ddof_0": [valid_reference[column].std(ddof=0) for column in player_metric_columns],
            "reference_count": [valid_reference[column].notna().sum() for column in player_metric_columns],
            "reference_true_shooting_percentage": (reference_true_shooting_percentage),
            "role_capacity_full_minutes_per_game": (ROLE_CAPACITY_FULL_MINUTES_PER_GAME),
            "role_capacity_exponent": (ROLE_CAPACITY_EXPONENT),
        }
    )

    invalid_std = parameter_table["std_ddof_0"].isna() | parameter_table["std_ddof_0"].le(0)
    if invalid_std.any():
        raise ValueError("Invalid metric standard deviations: " f"{parameter_table.loc[invalid_std, 'metric'].tolist()}")

    return parameter_table


def add_player_quality_scores(player_rows, metric_reference, reliability_shrinkage_constant, zscore_clip_limit=ZSCORE_CLIP_LIMIT):
    if reliability_shrinkage_constant < 0:
        raise ValueError("reliability_shrinkage_constant cannot be negative.")
    if zscore_clip_limit <= 0:
        raise ValueError("zscore_clip_limit must be positive.")

    reference = metric_reference.set_index("metric")
    reference_ts_values = metric_reference["reference_true_shooting_percentage"].dropna().unique()
    if len(reference_ts_values) != 1:
        raise ValueError("Metric reference must contain exactly one development-period " "true-shooting baseline.")

    reference_true_shooting_percentage = float(reference_ts_values[0])

    result = prepare_player_rate_metrics(player_rows, reference_true_shooting_percentage=(reference_true_shooting_percentage))

    qualified = qualified_rate_player_mask(result)
    below_threshold = below_minimum_minutes_mask(result)
    no_prior = known_no_prior_production_mask(result)

    zscore_columns = []

    for metric in player_metric_columns:
        if metric not in reference.index:
            raise KeyError(f"Metric reference is missing: {metric}")

        z_column = f"{metric}_zscore"
        result[z_column] = (result[metric] - float(reference.loc[metric, "mean"])) / float(reference.loc[metric, "std_ddof_0"])
        result.loc[~qualified, z_column] = np.nan
        zscore_columns.append(z_column)

    result[zscore_columns] = result[zscore_columns].clip(lower=-float(zscore_clip_limit), upper=float(zscore_clip_limit))

    for domain_name, metric_weights in player_domain_metric_weights.items():
        domain_column = f"{domain_name}_quality_score"
        result[domain_column] = 0.0
        result.loc[~qualified, domain_column] = np.nan

        for z_column, signed_weight in metric_weights.items():
            result.loc[qualified, domain_column] += float(signed_weight) * result.loc[qualified, z_column]

    result["season_player_raw_quality_score"] = 0.0
    result.loc[~qualified, "season_player_raw_quality_score"] = np.nan

    for domain_name in player_domains:
        domain_column = f"{domain_name}_quality_score"
        result.loc[qualified, "season_player_raw_quality_score"] += (
            float(player_domain_weights[domain_name]) * result.loc[qualified, domain_column]
        )

    possessions = pd.to_numeric(result["season_estimated_player_possessions"], errors="coerce")

    result["season_player_reliability"] = np.nan
    result.loc[qualified, "season_player_reliability"] = possessions.loc[qualified] / (
        possessions.loc[qualified] + float(reliability_shrinkage_constant)
    )

    result["season_player_shrunk_quality_score"] = np.nan
    result.loc[qualified, "season_player_shrunk_quality_score"] = (
        result.loc[qualified, "season_player_reliability"] * result.loc[qualified, "season_player_raw_quality_score"]
    )

    result["season_player_role_adjusted_quality_score"] = np.nan
    result.loc[qualified, "season_player_role_adjusted_quality_score"] = (
        result.loc[qualified, "season_player_role_capacity"] * result.loc[qualified, "season_player_shrunk_quality_score"]
    )

    # Use the metric name expected by the scoring calculation.
    result["season_player_quality_score"] = result["season_player_role_adjusted_quality_score"]

    result["reliability_shrinkage_constant"] = float(reliability_shrinkage_constant)
    result["zscore_clip_limit"] = float(zscore_clip_limit)

    result["player_value_status"] = "unresolved_or_incomplete"
    result.loc[qualified, "player_value_status"] = "calculated_efficiency_load_role_adjusted"
    result.loc[below_threshold, "player_value_status"] = "below_100_minute_threshold"
    result.loc[no_prior, "player_value_status"] = "known_zero_no_prior_nba_production"

    return result


def add_linear_player_production_value(
    quality_rows,
    base_player_value=BASE_PLAYER_VALUE,
    minimum_player_base=MINIMUM_PLAYER_BASE,
    quality_multiplier_slope=QUALITY_MULTIPLIER_SLOPE,
):
    """
    Convert finalized rate quality into a nonnegative player value.

    The full base value is reserved for players who demonstrate a full
    rotation role. Lower-role players receive an interpolated baseline:

        minimum base
        + (full base - minimum base) * role capacity

    The quality component remains reliability- and role-adjusted upstream.
    """
    if base_player_value < 0:
        raise ValueError("base_player_value cannot be negative.")
    if minimum_player_base < 0:
        raise ValueError("minimum_player_base cannot be negative.")
    if minimum_player_base > base_player_value:
        raise ValueError("minimum_player_base cannot exceed base_player_value.")
    if quality_multiplier_slope < 0:
        raise ValueError("quality_multiplier_slope cannot be negative.")

    result = quality_rows.copy()

    calculated = result["player_value_status"].eq("calculated_efficiency_load_role_adjusted")
    known_zero_no_prior = result["player_value_status"].eq("known_zero_no_prior_nba_production")

    quality = pd.to_numeric(result["season_player_quality_score"], errors="coerce")
    role_capacity = pd.to_numeric(result["season_player_role_capacity"], errors="coerce").clip(lower=0.0, upper=1.0)

    calculated &= quality.notna() & role_capacity.notna()

    result["season_role_adjusted_base_value"] = np.nan
    result.loc[calculated, "season_role_adjusted_base_value"] = (
        float(minimum_player_base) + (float(base_player_value) - float(minimum_player_base)) * role_capacity.loc[calculated]
    )
    result.loc[known_zero_no_prior, "season_role_adjusted_base_value"] = 0.0

    result["season_quality_component"] = np.nan
    result.loc[calculated, "season_quality_component"] = float(quality_multiplier_slope) * quality.loc[calculated]
    result.loc[known_zero_no_prior, "season_quality_component"] = 0.0

    result["season_player_production_value"] = np.nan
    result.loc[calculated, "season_player_production_value"] = np.clip(
        result.loc[calculated, "season_role_adjusted_base_value"] + result.loc[calculated, "season_quality_component"], 0.0, None
    )
    result.loc[known_zero_no_prior, "season_player_production_value"] = 0.0

    result["base_player_value"] = float(base_player_value)
    result["minimum_player_base"] = float(minimum_player_base)

    result["player_value_known"] = calculated | known_zero_no_prior
    result.loc[~result["player_value_known"] & ~result["player_value_status"].eq("below_100_minute_threshold"), "player_value_status"] = (
        "unresolved_or_incomplete"
    )

    if result["season_player_production_value"].dropna().lt(0).any():
        raise AssertionError("Individual player values must be nonnegative.")

    return result

## Apply a rotation-slot cost when combining multiple players


In [7]:
# Calculate package value after accounting for additional roster slots.
def calculate_slot_adjusted_package_value(individual_values, additional_player_slot_cost):
    values = pd.to_numeric(pd.Series(individual_values), errors="coerce")

    if values.empty:
        return 0.0
    if values.isna().any():
        return np.nan
    if (values < 0).any():
        raise ValueError("Individual player values cannot be negative.")

    ordered_values = np.sort(values.to_numpy(dtype="float64"))[::-1]

    leading_player_value = float(ordered_values[0])
    additional_player_values = ordered_values[1:]

    additional_contribution = np.clip(additional_player_values - float(additional_player_slot_cost), 0.0, None).sum()

    return float(leading_player_value + additional_contribution)


def aggregate_player_values_to_transactions(model_rows, player_value_rows, additional_player_slot_cost):
    if additional_player_slot_cost < 0:
        raise ValueError("additional_player_slot_cost cannot be negative.")

    result = model_rows.copy()
    player_rows = player_value_rows.copy()

    player_rows["transaction_side_clean"] = player_rows["transaction_side"].astype("string").str.strip().str.lower()

    unexpected_sides = set(player_rows["transaction_side_clean"].dropna()) - {"acquired", "relinquished"}
    if unexpected_sides:
        raise ValueError("Unexpected player transaction sides: " f"{sorted(unexpected_sides)}")

    player_rows["is_calculated_rate_value"] = (
        player_rows["player_value_status"].eq("calculated_efficiency_load_role_adjusted") & player_rows["player_value_known"]
    )
    player_rows["is_known_zero_no_prior"] = (
        player_rows["player_value_status"].eq("known_zero_no_prior_nba_production") & player_rows["player_value_known"]
    )
    player_rows["is_below_minute_threshold"] = player_rows["player_value_status"].eq("below_100_minute_threshold")

    side_summaries = []

    for side in ["acquired", "relinquished"]:
        side_rows = player_rows.loc[player_rows["transaction_side_clean"].eq(side)].copy()

        grouped = side_rows.groupby("transaction_row_id", sort=False)

        count_summary = grouped.agg(
            observed_player_row_count=("transaction_player_name", "size"),
            known_player_value_count=("player_value_known", "sum"),
            calculated_rate_player_count=("is_calculated_rate_value", "sum"),
            known_zero_no_prior_count=("is_known_zero_no_prior", "sum"),
            below_minute_threshold_count=("is_below_minute_threshold", "sum"),
            raw_player_production_value_sum=("season_player_production_value", lambda values: values.sum(min_count=1)),
            leading_player_value=("season_player_production_value", "max"),
        ).reset_index()

        adjusted_values = (
            grouped["season_player_production_value"]
            .apply(
                lambda values: (calculate_slot_adjusted_package_value(values, additional_player_slot_cost=(additional_player_slot_cost)))
            )
            .rename("slot_adjusted_player_production_value")
            .reset_index()
        )

        side_summary = count_summary.merge(adjusted_values, on="transaction_row_id", how="left", validate="one_to_one")

        side_summary["slot_cost_deduction"] = (
            side_summary["raw_player_production_value_sum"] - side_summary["slot_adjusted_player_production_value"]
        )

        rename_map = {column: f"{side}_{column}" for column in side_summary.columns if column != "transaction_row_id"}
        side_summary = side_summary.rename(columns=rename_map)
        side_summaries.append(side_summary)

    for side_summary in side_summaries:
        result = result.merge(side_summary, on="transaction_row_id", how="left", validate="one_to_one")

    for side in ["acquired", "relinquished"]:
        listed_column = f"{side}_listed_player_count"
        observed_column = f"{side}_observed_player_row_count"
        known_column = f"{side}_known_player_value_count"
        calculated_column = f"{side}_calculated_rate_player_count"
        zero_column = f"{side}_known_zero_no_prior_count"
        below_column = f"{side}_below_minute_threshold_count"
        raw_value_column = f"{side}_raw_player_production_value_sum"
        adjusted_value_column = f"{side}_slot_adjusted_player_production_value"
        leader_column = f"{side}_leading_player_value"
        deduction_column = f"{side}_slot_cost_deduction"
        complete_column = f"{side}_player_value_complete"

        if listed_column not in result.columns:
            raise KeyError("Model data is missing player coverage column: " f"{listed_column}")

        result[listed_column] = pd.to_numeric(result[listed_column], errors="coerce").fillna(0).astype("int64")

        for count_column in [observed_column, known_column, calculated_column, zero_column, below_column]:
            result[count_column] = pd.to_numeric(result[count_column], errors="coerce").fillna(0).astype("int64")

        result[complete_column] = result[observed_column].eq(result[listed_column]) & result[known_column].eq(result[listed_column])

        empty_side = result[listed_column].eq(0)
        for value_column in [raw_value_column, adjusted_value_column, leader_column, deduction_column]:
            result.loc[empty_side, value_column] = 0.0
            result.loc[~result[complete_column], value_column] = np.nan

    result["additional_player_slot_cost"] = float(additional_player_slot_cost)

    result["net_raw_player_production_value"] = (
        result["acquired_raw_player_production_value_sum"] - result["relinquished_raw_player_production_value_sum"]
    )

    result["net_player_production_value"] = (
        result["acquired_slot_adjusted_player_production_value"] - result["relinquished_slot_adjusted_player_production_value"]
    )

    result["player_value_target_eligible"] = (
        result["acquired_player_value_complete"]
        & result["relinquished_player_value_complete"]
        & result["net_player_production_value"].notna()
    )

    return result

## Prepare calibration samples and evaluation helpers


In [8]:
# Set the production target and constrained relative pick curve.
raw_target_column = "net_player_production_value"
relative_weights = relative_pick_curve.to_numpy(dtype="float64")


def add_calibration_flags(data):
    result = data.copy()

    result["absolute_pick_units"] = result[outright_pick_net_columns].abs().sum(axis=1)
    result["has_outright_pick_activity"] = result["absolute_pick_units"].gt(0)

    acquired_count = result["acquired_listed_player_count"]
    relinquished_count = result["relinquished_listed_player_count"]

    result["equal_nonzero_player_counts"] = acquired_count.eq(relinquished_count) & acquired_count.gt(0)

    result["one_for_one_player_package"] = (
        acquired_count.eq(1)
        & relinquished_count.eq(1)
        & result["acquired_calculated_rate_player_count"].eq(1)
        & result["relinquished_calculated_rate_player_count"].eq(1)
    )

    result["one_calculated_player_for_zero_players"] = (
        acquired_count.eq(1) & relinquished_count.eq(0) & result["acquired_calculated_rate_player_count"].eq(1)
    ) | (acquired_count.eq(0) & relinquished_count.eq(1) & result["relinquished_calculated_rate_player_count"].eq(1))

    result["one_player_vs_multiple_players"] = (acquired_count.eq(1) & relinquished_count.ge(2)) | (
        acquired_count.ge(2) & relinquished_count.eq(1)
    )

    result["has_multi_player_side"] = pd.concat([acquired_count, relinquished_count], axis=1).max(axis=1).ge(2)

    result["one_calculated_player_for_zero_with_picks"] = (
        result["one_calculated_player_for_zero_players"] & result["has_outright_pick_activity"]
    )

    result["one_player_vs_multiple_players_with_picks"] = result["one_player_vs_multiple_players"] & result["has_outright_pick_activity"]

    result["multi_player_packages_with_picks"] = result["has_multi_player_side"] & result["has_outright_pick_activity"]

    result["unequal_player_counts_with_picks"] = acquired_count.ne(relinquished_count) & result["has_outright_pick_activity"]

    result["one_for_one_with_picks"] = result["one_for_one_player_package"] & result["has_outright_pick_activity"]

    result["all_complete_packages_with_picks"] = result["has_outright_pick_activity"]

    return result


def prepare_model_period(scored_rows, seasons):
    eligible = scored_rows.loc[scored_rows["player_value_target_eligible"]].copy()

    eligible["has_swap"] = eligible[swap_net_columns].abs().sum(axis=1).gt(0)

    period = eligible.loc[~eligible["has_swap"] & eligible["production_reference_season"].isin(seasons)].copy().reset_index(drop=True)

    return add_calibration_flags(period)


def evaluate_predictions(model_name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")

    return {
        "model": model_name,
        "row_count": int(len(y_true)),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r_squared": r2_score(y_true, y_pred),
        "prediction_mean": float(y_pred.mean()),
        "prediction_standard_deviation": float(y_pred.std(ddof=0)),
    }


def fit_anchored_pick_scale_only(development_subset, relative_weights):
    required_columns = [raw_target_column, *outright_pick_net_columns]

    development_fit = development_subset[required_columns].dropna().copy()

    if development_fit.empty:
        raise ValueError("No development rows are available " "for pick-scale fitting.")

    X_development = development_fit[outright_pick_net_columns].astype("float64").to_numpy()
    y_development = development_fit[raw_target_column].astype("float64").to_numpy()

    relative_weight_array = np.asarray(relative_weights, dtype="float64")
    development_pick_index = X_development @ relative_weight_array

    denominator = float(np.dot(development_pick_index, development_pick_index))
    if denominator <= 0:
        raise ValueError("The pick index has no variation.")

    least_squares_scale = max(0.0, float(-np.dot(development_pick_index, y_development) / denominator))
    upper_bound = max(10.0, 10.0 * least_squares_scale)

    optimization = minimize_scalar(
        lambda scale: mean_absolute_error(y_development, -float(scale) * development_pick_index),
        bounds=(0.0, upper_bound),
        method="bounded",
        options={"xatol": 1e-10},
    )

    if not optimization.success:
        raise RuntimeError("Scale fitting failed: " f"{optimization.message}")

    scale = float(optimization.x)
    return {"development_rows": int(len(development_fit)), "selected_scale": scale, "weights": (scale * relative_weight_array)}


def evaluate_fixed_pick_weights(data_subset, model_name, pick_weights):
    required_columns = [raw_target_column, *outright_pick_net_columns]
    evaluation_data = data_subset[required_columns].dropna().copy()

    if evaluation_data.empty:
        return {
            "model": model_name,
            "row_count": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r_squared": np.nan,
            "baseline_mae": np.nan,
            "mae_ratio_to_zero_baseline": np.nan,
            "target_standard_deviation": np.nan,
        }

    X = evaluation_data[outright_pick_net_columns].astype("float64").to_numpy()
    y = evaluation_data[raw_target_column].astype("float64").to_numpy()
    predictions = -X @ np.asarray(pick_weights, dtype="float64")
    baseline = np.zeros_like(y)

    metrics = evaluate_predictions(model_name, y, predictions)
    baseline_mae = mean_absolute_error(y, baseline)
    metrics["baseline_mae"] = baseline_mae
    metrics["mae_ratio_to_zero_baseline"] = metrics["mae"] / baseline_mae if baseline_mae > 0 else np.nan
    metrics["target_standard_deviation"] = float(y.std(ddof=0))
    return metrics


def weighted_validation_score(sample_results):
    weighted_sum = 0.0
    used_weight = 0.0

    for sample_flag, sample_weight in VALIDATION_SAMPLE_WEIGHTS.items():
        result = sample_results[sample_flag]
        ratio = result["mae_ratio_to_zero_baseline"]

        if np.isfinite(ratio):
            weighted_sum += float(sample_weight) * float(ratio)
            used_weight += float(sample_weight)

    if used_weight <= 0:
        return np.nan

    return weighted_sum / used_weight


def fit_regularized_anchored_pick_scale_only(development_subset, relative_weights, prior_scale, regularization_strength):
    """Fit one nonnegative curve scale with a prior-scale penalty.

    The data-loss term is divided by the zero-prediction MAE so the
    regularization strength is dimensionless and comparable across player-value
    scales.
    """
    if prior_scale <= 0:
        raise ValueError("prior_scale must be positive.")
    if regularization_strength < 0:
        raise ValueError("regularization_strength cannot be negative.")

    required_columns = [raw_target_column, *outright_pick_net_columns]
    development_fit = development_subset[required_columns].dropna().copy()

    if development_fit.empty:
        raise ValueError("No development rows are available for regularized scale fitting.")

    X_development = development_fit[outright_pick_net_columns].astype("float64").to_numpy()
    y_development = development_fit[raw_target_column].astype("float64").to_numpy()
    relative_weight_array = np.asarray(relative_weights, dtype="float64")
    development_pick_index = X_development @ relative_weight_array

    if not np.any(np.abs(development_pick_index) > 0):
        raise ValueError("The pick index has no variation.")

    baseline_mae = float(mean_absolute_error(y_development, np.zeros_like(y_development)))
    if baseline_mae <= 0:
        raise ValueError("The training target has zero absolute scale.")

    unregularized_fit = fit_anchored_pick_scale_only(development_subset, relative_weights)
    unregularized_scale = float(unregularized_fit["selected_scale"])

    upper_bound = max(10.0, 5.0 * float(prior_scale), 5.0 * unregularized_scale)

    def objective(candidate_scale):
        candidate_scale = float(candidate_scale)
        predictions = -candidate_scale * development_pick_index
        data_mae = float(mean_absolute_error(y_development, predictions))
        normalized_data_mae = data_mae / baseline_mae
        relative_scale_change = (candidate_scale - float(prior_scale)) / float(prior_scale)
        penalty = float(regularization_strength) * relative_scale_change**2
        return normalized_data_mae + penalty

    optimization = minimize_scalar(objective, bounds=(0.0, upper_bound), method="bounded", options={"xatol": 1e-10})

    if not optimization.success:
        raise RuntimeError("Regularized scale fitting failed: " f"{optimization.message}")

    selected_scale = float(optimization.x)
    predictions = -selected_scale * development_pick_index
    training_mae = float(mean_absolute_error(y_development, predictions))
    normalized_training_mae = training_mae / baseline_mae
    relative_scale_change = (selected_scale - float(prior_scale)) / float(prior_scale)
    regularization_penalty = float(regularization_strength) * relative_scale_change**2

    return {
        "development_rows": int(len(development_fit)),
        "prior_scale": float(prior_scale),
        "unregularized_scale": unregularized_scale,
        "regularization_strength": float(regularization_strength),
        "selected_scale": selected_scale,
        "relative_scale_change": relative_scale_change,
        "percentage_scale_change": 100.0 * relative_scale_change,
        "training_baseline_mae": baseline_mae,
        "training_mae": training_mae,
        "normalized_training_mae": normalized_training_mae,
        "regularization_penalty": regularization_penalty,
        "regularized_objective": (normalized_training_mae + regularization_penalty),
        "weights": (selected_scale * relative_weight_array),
    }


def weighted_pick_scale_validation_score(sample_results):
    weighted_sum = 0.0
    used_weight = 0.0

    for sample_flag, sample_weight in PICK_SCALE_VALIDATION_SAMPLE_WEIGHTS.items():
        sample_result = sample_results[sample_flag]
        ratio = sample_result["mae_ratio_to_zero_baseline"]
        if np.isfinite(ratio):
            weighted_sum += float(sample_weight) * float(ratio)
            used_weight += float(sample_weight)

    if used_weight <= 0:
        return np.nan

    return weighted_sum / used_weight

## Jointly tune reliability shrinkage and the additional-player slot cost

The development-period shooting baseline, z-score means, and z-score standard deviations are calculated using parameter-training seasons only. The validation seasons select the possession-reliability constant and package slot cost. The role-capacity formula is fixed before this search.


In [9]:
# Build the parameter-training player reference sample.
parameter_reference_player_rows = transaction_player_data.loc[
    transaction_player_data["production_reference_season"].isin(parameter_training_seasons)
].copy()

parameter_metric_reference = build_player_metric_reference(parameter_reference_player_rows)

hyperparameter_records = []

for shrinkage_constant in CANDIDATE_RELIABILITY_SHRINKAGE_CONSTANTS:
    parameter_quality_rows = add_player_quality_scores(
        transaction_player_data, metric_reference=(parameter_metric_reference), reliability_shrinkage_constant=(shrinkage_constant)
    )

    parameter_player_value_rows = add_linear_player_production_value(
        parameter_quality_rows, base_player_value=BASE_PLAYER_VALUE, quality_multiplier_slope=(QUALITY_MULTIPLIER_SLOPE)
    )

    qualified_training_rows = parameter_player_value_rows.loc[
        parameter_player_value_rows["production_reference_season"].isin(parameter_training_seasons)
        & parameter_player_value_rows["player_value_status"].eq("calculated_efficiency_load_role_adjusted")
    ].copy()

    qualified_training_player_values = qualified_training_rows["season_player_production_value"].dropna().astype("float64")

    if qualified_training_player_values.empty:
        raise ValueError("No qualified training-period player values " "are available for hyperparameter diagnostics.")

    training_value_distribution = qualified_training_player_values.describe(percentiles=[0.10, 0.50, 0.90, 0.95, 0.99])

    training_median_value = float(training_value_distribution["50%"])
    training_p95_value = float(training_value_distribution["95%"])

    reliability_values = qualified_training_rows["season_player_reliability"].dropna().astype("float64")

    median_training_reliability = float(reliability_values.median())

    low_reliability_rows = qualified_training_rows.loc[qualified_training_rows["season_player_reliability"].le(median_training_reliability)]
    high_reliability_rows = qualified_training_rows.loc[
        qualified_training_rows["season_player_reliability"].gt(median_training_reliability)
    ]

    low_reliability_p95_value = float(low_reliability_rows["season_player_production_value"].quantile(0.95))
    high_reliability_p95_value = float(high_reliability_rows["season_player_production_value"].quantile(0.95))

    role_capacity_values = qualified_training_rows["season_player_role_capacity"].dropna().astype("float64")
    median_training_role_capacity = float(role_capacity_values.median())
    low_role_rows = qualified_training_rows.loc[qualified_training_rows["season_player_role_capacity"].le(median_training_role_capacity)]
    high_role_rows = qualified_training_rows.loc[qualified_training_rows["season_player_role_capacity"].gt(median_training_role_capacity)]
    low_role_p95_value = float(low_role_rows["season_player_production_value"].quantile(0.95))
    high_role_p95_value = float(high_role_rows["season_player_production_value"].quantile(0.95))

    for slot_cost in CANDIDATE_ADDITIONAL_PLAYER_SLOT_COSTS:
        candidate_scored_rows = aggregate_player_values_to_transactions(
            draft_weight_model_data, parameter_player_value_rows, additional_player_slot_cost=(slot_cost)
        )

        candidate_training_data = prepare_model_period(candidate_scored_rows, parameter_training_seasons)
        candidate_validation_data = prepare_model_period(candidate_scored_rows, validation_seasons)

        fit_subset = candidate_training_data.loc[candidate_training_data["one_calculated_player_for_zero_with_picks"]]

        scale_result = fit_anchored_pick_scale_only(fit_subset, relative_weights)
        candidate_pick_weights = scale_result["weights"]

        sample_results = {}
        for sample_flag in VALIDATION_SAMPLE_WEIGHTS:
            sample_results[sample_flag] = evaluate_fixed_pick_weights(
                candidate_validation_data.loc[candidate_validation_data[sample_flag]],
                model_name=sample_flag,
                pick_weights=(candidate_pick_weights),
            )

        two_median_package_value = calculate_slot_adjusted_package_value(
            [training_median_value, training_median_value], additional_player_slot_cost=(slot_cost)
        )

        second_median_contribution = max(training_median_value - float(slot_cost), 0.0)
        second_median_contribution_share = second_median_contribution / training_median_value if training_median_value > 0 else np.nan

        p95_to_two_median_package_ratio = training_p95_value / two_median_package_value if two_median_package_value > 0 else np.inf

        face_validity_ok = (
            p95_to_two_median_package_ratio >= MINIMUM_P95_TO_TWO_MEDIAN_PACKAGE_RATIO
            and second_median_contribution_share >= MINIMUM_SECOND_MEDIAN_CONTRIBUTION_SHARE
        )

        record = {
            "reliability_shrinkage_constant": float(shrinkage_constant),
            "additional_player_slot_cost": float(slot_cost),
            "training_scale_rows": int(scale_result["development_rows"]),
            "selected_pick_curve_scale": float(scale_result["selected_scale"]),
            "validation_score": float(weighted_validation_score(sample_results)),
            "training_value_mean": float(qualified_training_player_values.mean()),
            "training_value_standard_deviation": float(qualified_training_player_values.std(ddof=0)),
            "training_median_player_value": (training_median_value),
            "training_p95_player_value": (training_p95_value),
            "training_median_reliability": (median_training_reliability),
            "training_10th_percentile_reliability": float(reliability_values.quantile(0.10)),
            "training_90th_percentile_reliability": float(reliability_values.quantile(0.90)),
            "low_reliability_p95_player_value": (low_reliability_p95_value),
            "high_reliability_p95_player_value": (high_reliability_p95_value),
            "low_to_high_reliability_p95_ratio": (
                low_reliability_p95_value / high_reliability_p95_value if high_reliability_p95_value > 0 else np.nan
            ),
            "training_median_role_capacity": (median_training_role_capacity),
            "training_10th_percentile_role_capacity": float(role_capacity_values.quantile(0.10)),
            "training_90th_percentile_role_capacity": float(role_capacity_values.quantile(0.90)),
            "low_role_p95_player_value": (low_role_p95_value),
            "high_role_p95_player_value": (high_role_p95_value),
            "low_to_high_role_p95_ratio": (low_role_p95_value / high_role_p95_value if high_role_p95_value > 0 else np.nan),
            "two_median_package_value": (two_median_package_value),
            "second_median_contribution": (second_median_contribution),
            "second_median_contribution_share": (second_median_contribution_share),
            "p95_to_two_median_package_ratio": (p95_to_two_median_package_ratio),
            "face_validity_ok": bool(face_validity_ok),
        }

        for sample_flag, sample_result in sample_results.items():
            prefix = sample_flag.replace("_with_picks", "")
            record[f"{prefix}_validation_rows"] = int(sample_result["row_count"])
            record[f"{prefix}_mae"] = float(sample_result["mae"]) if np.isfinite(sample_result["mae"]) else np.nan
            record[f"{prefix}_mae_ratio"] = (
                float(sample_result["mae_ratio_to_zero_baseline"]) if np.isfinite(sample_result["mae_ratio_to_zero_baseline"]) else np.nan
            )
            record[f"{prefix}_r_squared"] = float(sample_result["r_squared"]) if np.isfinite(sample_result["r_squared"]) else np.nan

        hyperparameter_records.append(record)

hyperparameter_search = pd.DataFrame(hyperparameter_records)

selection_pool = hyperparameter_search.loc[
    hyperparameter_search["validation_score"].notna() & hyperparameter_search["face_validity_ok"]
].copy()

if selection_pool.empty:
    print("Warning: no candidate passed both face-validity " "constraints. Selection will fall back to validation " "performance only.")
    selection_pool = hyperparameter_search.loc[hyperparameter_search["validation_score"].notna()].copy()

if selection_pool.empty:
    raise ValueError("No hyperparameter candidate produced a valid " "internal-validation score.")

best_validation_score = float(selection_pool["validation_score"].min())

near_best_candidates = selection_pool.loc[selection_pool["validation_score"].le(best_validation_score + VALIDATION_SCORE_TOLERANCE)].copy()

# When validation performance is effectively tied, prefer more conservative
# reliability shrinkage, then the less aggressive slot penalty.
selected_hyperparameter_row = near_best_candidates.sort_values(
    ["reliability_shrinkage_constant", "additional_player_slot_cost", "validation_score"], ascending=[False, True, True]
).iloc[0]

SELECTED_RELIABILITY_SHRINKAGE_CONSTANT = float(selected_hyperparameter_row["reliability_shrinkage_constant"])

SELECTED_ADDITIONAL_PLAYER_SLOT_COST = float(selected_hyperparameter_row["additional_player_slot_cost"])

display(
    hyperparameter_search.sort_values(
        ["face_validity_ok", "validation_score", "reliability_shrinkage_constant", "additional_player_slot_cost"],
        ascending=[False, True, False, True],
    )
)

print("Selected reliability shrinkage constant:", SELECTED_RELIABILITY_SHRINKAGE_CONSTANT)
print("Selected additional-player slot cost:", SELECTED_ADDITIONAL_PLAYER_SLOT_COST)
print("Best eligible validation score:", best_validation_score)
print("Selected validation score:", float(selected_hyperparameter_row["validation_score"]))

,reliability_shrinkage_constant,additional_player_slot_cost,training_scale_rows,selected_pick_curve_scale,validation_score,training_value_mean,training_value_standard_deviation,training_median_player_value,training_p95_player_value,training_median_reliability,...,unequal_player_counts_mae_ratio,unequal_player_counts_r_squared,multi_player_packages_validation_rows,multi_player_packages_mae,multi_player_packages_mae_ratio,multi_player_packages_r_squared,all_complete_packages_validation_rows,all_complete_packages_mae,all_complete_packages_mae_ratio,all_complete_packages_r_squared
2,250.0,0.50,241,1.332336,2.052430,0.874452,0.242521,0.857971,1.310415,0.885708,...,1.252778,-1.556684,55,1.118229,2.694425,-9.099064,148,0.804783,1.319516,-1.635321
7,500.0,0.50,241,1.319896,2.115816,0.875990,0.229445,0.858690,1.291083,0.794862,...,1.260788,-1.582353,55,1.114135,2.748814,-9.311839,148,0.805707,1.329636,-1.663773
12,750.0,0.50,241,1.308723,2.150002,0.876352,0.220019,0.857898,1.275221,0.720918,...,1.265592,-1.594723,55,1.109307,2.780530,-9.383948,148,0.804917,1.335583,-1.677619
17,1000.0,0.50,241,1.298633,2.165327,0.876199,0.212644,0.857242,1.259784,0.659561,...,1.268120,-1.605266,55,1.104756,2.793722,-9.403165,148,0.803498,1.338686,-1.689046
3,250.0,0.75,241,1.332336,2.567796,0.874452,0.242521,0.857971,1.310415,0.885708,...,1.305146,-1.588000,55,1.041313,3.559040,-14.534860,148,0.776199,1.375243,-1.676800
8,500.0,0.75,241,1.319896,2.670569,0.875990,0.229445,0.858690,1.291083,0.794862,...,1.311675,-1.591351,55,1.031190,3.671906,-15.095271,148,0.774883,1.384461,-1.683310
13,750.0,0.75,241,1.308723,2.737151,0.876352,0.220019,0.857898,1.275221,0.720918,...,1.315731,-1.595982,55,1.023587,3.743727,-15.534530,148,0.773062,1.390356,-1.689557
18,1000.0,0.75,241,1.298633,2.789478,0.876199,0.212644,0.857242,1.259784,0.659561,...,1.318792,-1.601027,55,1.016746,3.799185,-15.895830,148,0.770792,1.394560,-1.695613
15,1000.0,0.00,241,1.298633,1.351081,0.876199,0.212644,0.857242,1.259784,0.659561,...,1.109026,-1.085973,55,1.498229,1.540266,-2.086226,148,0.949721,1.165676,-1.135679
10,750.0,0.00,241,1.308723,1.362998,0.876352,0.220019,0.857898,1.275221,0.720918,...,1.112466,-1.093729,55,1.498909,1.556082,-2.128048,148,0.949702,1.169039,-1.143629


Selected reliability shrinkage constant: 250.0
Selected additional-player slot cost: 0.5
Best eligible validation score: 2.0524304399638624
Selected validation score: 2.0524304399638624


## Rebuild the final model using every development season

In [10]:
# Build the development-period player reference sample.
development_reference_player_rows = transaction_player_data.loc[
    transaction_player_data["production_reference_season"].isin(development_seasons)
].copy()

final_metric_reference = build_player_metric_reference(development_reference_player_rows)

all_quality_rows = add_player_quality_scores(
    transaction_player_data,
    metric_reference=(final_metric_reference),
    reliability_shrinkage_constant=(SELECTED_RELIABILITY_SHRINKAGE_CONSTANT),
)

all_player_value_rows = add_linear_player_production_value(
    all_quality_rows, base_player_value=BASE_PLAYER_VALUE, quality_multiplier_slope=(QUALITY_MULTIPLIER_SLOPE)
)

scored_model_data = aggregate_player_values_to_transactions(
    draft_weight_model_data, all_player_value_rows, additional_player_slot_cost=(SELECTED_ADDITIONAL_PLAYER_SLOT_COST)
)

development_data = prepare_model_period(scored_model_data, development_seasons)
test_data = prepare_model_period(scored_model_data, test_seasons)

status_audit = (
    all_player_value_rows["player_value_status"].value_counts(dropna=False).rename_axis("player_value_status").reset_index(name="row_count")
)

sample_flags = [
    "all_complete_packages_with_picks",
    "unequal_player_counts_with_picks",
    "multi_player_packages_with_picks",
    "one_player_vs_multiple_players_with_picks",
    "one_calculated_player_for_zero_with_picks",
    "one_for_one_with_picks",
]

sample_audit = pd.DataFrame(
    [
        {
            "sample": sample_flag,
            "development_rows": int(development_data[sample_flag].sum()),
            "test_rows": int(test_data[sample_flag].sum()),
        }
        for sample_flag in sample_flags
    ]
)

display(status_audit)
display(sample_audit)
display(final_metric_reference)

print("Selected reliability shrinkage constant:", SELECTED_RELIABILITY_SHRINKAGE_CONSTANT)
print("Selected additional-player slot cost:", SELECTED_ADDITIONAL_PLAYER_SLOT_COST)
print("Target-eligible transaction rows:", int(scored_model_data["player_value_target_eligible"].sum()))

,player_value_status,row_count
0,calculated_efficiency_load_role_adjusted,4449
1,known_zero_no_prior_nba_production,855
2,unresolved_or_incomplete,623
3,below_100_minute_threshold,541


,sample,development_rows,test_rows
0,all_complete_packages_with_picks,720,144
1,unequal_player_counts_with_picks,464,91
2,multi_player_packages_with_picks,177,39
3,one_player_vs_multiple_players_with_picks,96,24
4,one_calculated_player_for_zero_with_picks,313,59
5,one_for_one_with_picks,145,42


,metric,mean,std_ddof_0,reference_count,reference_true_shooting_percentage,role_capacity_full_minutes_per_game,role_capacity_exponent
0,season_true_shooting_attempts_per_100,18.980307,4.771070,1843,0.524785,30.0,0.5
1,season_efficiency_points_added_per_100,-0.596218,2.081103,1843,0.524785,30.0,0.5
2,season_assists_per_100,4.344317,2.963983,1843,0.524785,30.0,0.5
3,season_turnovers_per_100,3.178666,1.092659,1843,0.524785,30.0,0.5
4,season_total_rebound_percentage,9.960107,4.496149,1843,0.524785,30.0,0.5
5,season_steal_percentage,1.642136,0.691633,1843,0.524785,30.0,0.5
6,season_block_percentage,1.429022,1.461572,1843,0.524785,30.0,0.5
7,season_plus_minus_per_100,-2.173668,5.756884,1843,0.524785,30.0,0.5


Selected reliability shrinkage constant: 250.0
Selected additional-player slot cost: 0.5
Target-eligible transaction rows: 1373


## Select the pick-scale regularization strength on internal validation

The finalized player values and slot-adjusted packages are held fixed in this section.

Each candidate regularization strength is fit on parameter-training transactions containing one calculated player on one side, zero players on the other side, and at least one outright pick. Candidate scales are then evaluated on the internal validation period using:

- 50% one calculated player for zero with picks;
- 30% unequal player counts with picks;
- 20% all complete packages with picks.

The zero-strength candidate is an unregularized benchmark. Final model selection is restricted to positive strengths. When validation performance is effectively tied, the stronger regularization strength is preferred.


In [11]:
# Fit the regularized absolute scale for the fixed pick curve.
regularization_training_data = prepare_model_period(scored_model_data, parameter_training_seasons)
regularization_validation_data = prepare_model_period(scored_model_data, validation_seasons)

PRIMARY_PICK_SCALE_SAMPLE_FLAG = "one_calculated_player_for_zero_with_picks"

regularization_training_subset = regularization_training_data.loc[regularization_training_data[PRIMARY_PICK_SCALE_SAMPLE_FLAG]].copy()

if regularization_training_subset.empty:
    raise ValueError("The parameter-training period contains no primary pick-scale rows.")

regularization_search_records = []

for regularization_strength in CANDIDATE_PICK_SCALE_REGULARIZATION_STRENGTHS:
    fit_result = fit_regularized_anchored_pick_scale_only(
        regularization_training_subset,
        relative_weights,
        prior_scale=PRIOR_PICK_CURVE_SCALE,
        regularization_strength=(regularization_strength),
    )
    candidate_weights = np.asarray(fit_result["weights"], dtype="float64")

    sample_results = {}
    for sample_flag in PICK_SCALE_VALIDATION_SAMPLE_WEIGHTS:
        sample_results[sample_flag] = evaluate_fixed_pick_weights(
            regularization_validation_data.loc[regularization_validation_data[sample_flag]],
            model_name=sample_flag,
            pick_weights=candidate_weights,
        )

    record = {key: value for key, value in fit_result.items() if key != "weights"}
    record["validation_score"] = float(weighted_pick_scale_validation_score(sample_results))

    for sample_flag, sample_result in sample_results.items():
        prefix = sample_flag.replace("_with_picks", "")
        record[f"{prefix}_validation_rows"] = int(sample_result["row_count"])
        record[f"{prefix}_validation_mae"] = float(sample_result["mae"]) if np.isfinite(sample_result["mae"]) else np.nan
        record[f"{prefix}_validation_mae_ratio"] = (
            float(sample_result["mae_ratio_to_zero_baseline"]) if np.isfinite(sample_result["mae_ratio_to_zero_baseline"]) else np.nan
        )
        record[f"{prefix}_validation_r_squared"] = float(sample_result["r_squared"]) if np.isfinite(sample_result["r_squared"]) else np.nan

    regularization_search_records.append(record)

regularization_search = pd.DataFrame(regularization_search_records)

positive_regularization_pool = regularization_search.loc[
    regularization_search["regularization_strength"].gt(0) & regularization_search["validation_score"].notna()
].copy()

if positive_regularization_pool.empty:
    raise ValueError("No positive regularization strength produced a validation score.")

best_regularized_validation_score = float(positive_regularization_pool["validation_score"].min())

near_best_regularized_candidates = positive_regularization_pool.loc[
    positive_regularization_pool["validation_score"].le(best_regularized_validation_score + PICK_SCALE_VALIDATION_SCORE_TOLERANCE)
].copy()

selected_regularization_row = (
    near_best_regularized_candidates.assign(absolute_percentage_scale_change=lambda frame: (frame["percentage_scale_change"].abs()))
    .sort_values(["regularization_strength", "absolute_percentage_scale_change", "validation_score"], ascending=[False, True, True])
    .iloc[0]
)

SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH = float(selected_regularization_row["regularization_strength"])

unregularized_validation_row = regularization_search.loc[regularization_search["regularization_strength"].eq(0.0)]

prior_validation_results = {}
prior_pick_weights = PRIOR_PICK_CURVE_SCALE * np.asarray(relative_weights, dtype="float64")
for sample_flag in PICK_SCALE_VALIDATION_SAMPLE_WEIGHTS:
    prior_validation_results[sample_flag] = evaluate_fixed_pick_weights(
        regularization_validation_data.loc[regularization_validation_data[sample_flag]],
        model_name=sample_flag,
        pick_weights=prior_pick_weights,
    )
prior_validation_score = float(weighted_pick_scale_validation_score(prior_validation_results))

regularization_selection_summary = pd.DataFrame(
    [
        {
            "candidate": "Prior deployed scale",
            "regularization_strength": np.nan,
            "scale": PRIOR_PICK_CURVE_SCALE,
            "percentage_change_from_prior": 0.0,
            "validation_score": prior_validation_score,
        },
        {
            "candidate": "Unregularized fit",
            "regularization_strength": 0.0,
            "scale": float(unregularized_validation_row["selected_scale"].iloc[0]),
            "percentage_change_from_prior": float(unregularized_validation_row["percentage_scale_change"].iloc[0]),
            "validation_score": float(unregularized_validation_row["validation_score"].iloc[0]),
        },
        {
            "candidate": "Selected regularized fit",
            "regularization_strength": (SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH),
            "scale": float(selected_regularization_row["selected_scale"]),
            "percentage_change_from_prior": float(selected_regularization_row["percentage_scale_change"]),
            "validation_score": float(selected_regularization_row["validation_score"]),
        },
    ]
)

display(regularization_search.sort_values(["validation_score", "regularization_strength"], ascending=[True, False]))
display(regularization_selection_summary)

print("Selected pick-scale regularization strength:", SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH)
print("Selected internal-validation score:", float(selected_regularization_row["validation_score"]))

,development_rows,prior_scale,unregularized_scale,regularization_strength,selected_scale,relative_scale_change,percentage_scale_change,training_baseline_mae,training_mae,normalized_training_mae,...,one_calculated_player_for_zero_validation_mae_ratio,one_calculated_player_for_zero_validation_r_squared,unequal_player_counts_validation_rows,unequal_player_counts_validation_mae,unequal_player_counts_validation_mae_ratio,unequal_player_counts_validation_r_squared,all_complete_packages_validation_rows,all_complete_packages_validation_mae,all_complete_packages_validation_mae_ratio,all_complete_packages_validation_r_squared
0,241,1.415349,1.331908,0.0,1.331908,-0.058954,-5.895407,0.858312,0.519323,0.605051,...,0.756929,0.062055,121,0.893558,1.253186,-1.566740,148,0.804994,1.319675,-1.645081
1,241,1.415349,1.331908,0.1,1.331908,-0.058954,-5.895405,0.858312,0.519323,0.605051,...,0.756929,0.062055,121,0.893558,1.253186,-1.566740,148,0.804994,1.319675,-1.645081
2,241,1.415349,1.331908,0.3,1.354653,-0.042884,-4.288411,0.858312,0.519543,0.605308,...,0.757515,0.050356,121,0.900979,1.263593,-1.634670,148,0.812156,1.331416,-1.715851
3,241,1.415349,1.331908,1.0,1.392727,-0.015984,-1.598367,0.858312,0.520282,0.606168,...,0.758497,0.029812,121,0.913954,1.281790,-1.751479,148,0.824617,1.351845,-1.837528
4,241,1.415349,1.331908,3.0,1.407808,-0.005328,-0.532789,0.858312,0.520574,0.606509,...,0.758885,0.021342,121,0.919111,1.289022,-1.798820,148,0.829628,1.360059,-1.886837
5,241,1.415349,1.331908,10.0,1.413087,-0.001598,-0.159837,0.858312,0.520676,0.606628,...,0.759021,0.018333,121,0.920916,1.291553,-1.815534,148,0.831382,1.362933,-1.904244
6,241,1.415349,1.331908,30.0,1.414595,-0.000533,-0.053279,0.858312,0.520706,0.606662,...,0.759060,0.017469,121,0.921431,1.292276,-1.820323,148,0.831883,1.363755,-1.909232
7,241,1.415349,1.331908,100.0,1.415123,-0.000160,-0.015984,0.858312,0.520716,0.606674,...,0.759074,0.017167,121,0.921612,1.292529,-1.822000,148,0.832058,1.364042,-1.910979
8,241,1.415349,1.331908,300.0,1.415274,-0.000053,-0.005328,0.858312,0.520719,0.606677,...,0.759078,0.017080,121,0.921663,1.292602,-1.822480,148,0.832108,1.364124,-1.911478
9,241,1.415349,1.331908,1000.0,1.415326,-0.000016,-0.001598,0.858312,0.520720,0.606679,...,0.759079,0.017050,121,0.921681,1.292627,-1.822648,148,0.832126,1.364153,-1.911653


,candidate,regularization_strength,scale,percentage_change_from_prior,validation_score
0,Prior deployed scale,NaN,1.415349,0.000000,1.040164
1,Unregularized fit,0.0,1.331908,-5.895407,1.018355
2,Selected regularized fit,0.3,1.354653,-4.288411,1.024119


Selected pick-scale regularization strength: 0.3
Selected internal-validation score: 1.024118876689496


## Refit the selected regularized scale on every development season

The selected positive regularization strength is now held fixed. The scale is refit on all development-period one-player-for-zero transactions. The notebook also calculates:

- the prior deployed scale;
- the full-development unregularized scale;
- the deployed regularized scale;
- percentage movement from the prior;
- a bootstrap confidence interval for the deployed regularized scale.


In [12]:
# Evaluate the preferred model on its primary development sample.
primary_development_subset = development_data.loc[development_data[PRIMARY_PICK_SCALE_SAMPLE_FLAG]].copy()

if primary_development_subset.empty:
    raise ValueError("The full development period contains no primary pick-scale rows.")

full_development_unregularized_fit = fit_anchored_pick_scale_only(primary_development_subset, relative_weights)

full_development_regularized_fit = fit_regularized_anchored_pick_scale_only(
    primary_development_subset,
    relative_weights,
    prior_scale=PRIOR_PICK_CURVE_SCALE,
    regularization_strength=(SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH),
)

unregularized_pick_curve_scale = float(full_development_unregularized_fit["selected_scale"])
selected_pick_curve_scale = float(full_development_regularized_fit["selected_scale"])

unregularized_pick_weights = np.asarray(full_development_unregularized_fit["weights"], dtype="float64")
additive_pick_weights = np.asarray(full_development_regularized_fit["weights"], dtype="float64")

rng = np.random.default_rng(PICK_SCALE_BOOTSTRAP_RANDOM_SEED)
bootstrap_scale_records = []

for bootstrap_iteration in range(PICK_SCALE_BOOTSTRAP_RESAMPLES):
    sampled_positions = rng.integers(0, len(primary_development_subset), size=len(primary_development_subset))
    bootstrap_subset = primary_development_subset.iloc[sampled_positions].reset_index(drop=True)
    bootstrap_fit = fit_regularized_anchored_pick_scale_only(
        bootstrap_subset,
        relative_weights,
        prior_scale=PRIOR_PICK_CURVE_SCALE,
        regularization_strength=(SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH),
    )
    bootstrap_scale_records.append(
        {"bootstrap_iteration": bootstrap_iteration, "regularized_scale": float(bootstrap_fit["selected_scale"])}
    )

pick_scale_bootstrap = pd.DataFrame(bootstrap_scale_records)

bootstrap_scale_summary = pd.DataFrame(
    [
        {
            "bootstrap_resamples": int(PICK_SCALE_BOOTSTRAP_RESAMPLES),
            "bootstrap_mean": float(pick_scale_bootstrap["regularized_scale"].mean()),
            "bootstrap_median": float(pick_scale_bootstrap["regularized_scale"].median()),
            "bootstrap_2_5_percentile": float(pick_scale_bootstrap["regularized_scale"].quantile(0.025)),
            "bootstrap_97_5_percentile": float(pick_scale_bootstrap["regularized_scale"].quantile(0.975)),
        }
    ]
)

additive_pick_weight_table = pd.DataFrame(
    {
        "tier": outright_pick_hierarchy,
        "asset_family": "outright_pick",
        "relative_curve_weight": relative_weights,
        "prior_estimated_value": prior_pick_weights,
        "unregularized_estimated_value": (unregularized_pick_weights),
        "estimated_value": additive_pick_weights,
        "net_count_column": outright_pick_net_columns,
        "calibration_method": ("fixed_relative_curve_regularized_absolute_scale_" "one_calculated_player_for_zero"),
    }
)

additive_pick_weight_table["percentage_change_from_prior"] = (
    100.0
    * (additive_pick_weight_table["estimated_value"] - additive_pick_weight_table["prior_estimated_value"])
    / additive_pick_weight_table["prior_estimated_value"]
)

assert additive_pick_weight_table["estimated_value"].ge(0).all()
assert additive_pick_weight_table["estimated_value"].is_monotonic_increasing
assert additive_pick_weight_table["estimated_value"].notna().all()

pick_scale_audit = pd.DataFrame(
    [
        {"scale_type": "Prior deployed", "scale": PRIOR_PICK_CURVE_SCALE, "percentage_change_from_prior": 0.0},
        {
            "scale_type": "Full-development unregularized",
            "scale": unregularized_pick_curve_scale,
            "percentage_change_from_prior": (100.0 * (unregularized_pick_curve_scale - PRIOR_PICK_CURVE_SCALE) / PRIOR_PICK_CURVE_SCALE),
        },
        {
            "scale_type": "Full-development regularized",
            "scale": selected_pick_curve_scale,
            "percentage_change_from_prior": (100.0 * (selected_pick_curve_scale - PRIOR_PICK_CURVE_SCALE) / PRIOR_PICK_CURVE_SCALE),
        },
    ]
)

display(pick_scale_audit)
display(bootstrap_scale_summary)
display(additive_pick_weight_table)

print("Selected positive regularization strength:", SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH)
print("Prior pick-curve scale:", PRIOR_PICK_CURVE_SCALE)
print("Unregularized full-development scale:", unregularized_pick_curve_scale)
print("Regularized deployed scale:", selected_pick_curve_scale)

,scale_type,scale,percentage_change_from_prior
0,Prior deployed,1.415349,0.000000
1,Full-development unregularized,1.290606,-8.813577
2,Full-development regularized,1.354653,-4.288412


,bootstrap_resamples,bootstrap_mean,bootstrap_median,bootstrap_2_5_percentile,bootstrap_97_5_percentile
0,500,1.363289,1.354653,1.231927,1.493333


,tier,asset_family,relative_curve_weight,prior_estimated_value,unregularized_estimated_value,estimated_value,net_count_column,calibration_method,percentage_change_from_prior
0,projected_late_second,outright_pick,0.168931,0.239096,0.218023,0.228842,net_projected_late_second_count,fixed_relative_curve_regularized_absolute_scal...,-4.288412
1,projected_early_second,outright_pick,0.168931,0.239096,0.218023,0.228842,net_projected_early_second_count,fixed_relative_curve_regularized_absolute_scal...,-4.288412
2,projected_late_first,outright_pick,0.414182,0.586212,0.534546,0.561073,net_projected_late_first_count,fixed_relative_curve_regularized_absolute_scal...,-4.288412
3,projected_lottery_first,outright_pick,0.844753,1.195620,1.090243,1.144347,net_projected_lottery_first_count,fixed_relative_curve_regularized_absolute_scal...,-4.288412
4,projected_top_5_first,outright_pick,1.000000,1.415349,1.290606,1.354653,net_projected_top_5_first_count,fixed_relative_curve_regularized_absolute_scal...,-4.288412


Selected positive regularization strength: 0.3
Prior pick-curve scale: 1.415349
Unregularized full-development scale: 1.2906061299095215
Regularized deployed scale: 1.3546530053314645


## Evaluate prior, unregularized, and regularized scales on the untouched test period

The final test period is not used for player-metric references, player-model hyperparameter selection, regularization-strength selection, or scale fitting.

The primary application weights are the selected regularized values. Prior and unregularized results are retained as diagnostics.


In [13]:
# Calculate held-out errors for fixed pick-weight candidates.
def evaluate_fixed_pick_weights_with_errors(data_subset, model_name, pick_weights):
    metrics = evaluate_fixed_pick_weights(data_subset, model_name=model_name, pick_weights=pick_weights)

    required_columns = [raw_target_column, *outright_pick_net_columns]
    evaluation_data = data_subset[required_columns].dropna().copy()

    if evaluation_data.empty:
        return {
            **metrics,
            "25th_percentile_absolute_error": np.nan,
            "median_absolute_error": np.nan,
            "75th_percentile_absolute_error": np.nan,
            "90th_percentile_absolute_error": np.nan,
        }

    X = evaluation_data[outright_pick_net_columns].astype("float64").to_numpy()
    y = evaluation_data[raw_target_column].astype("float64").to_numpy()
    predictions = -X @ np.asarray(pick_weights, dtype="float64")
    absolute_errors = np.abs(y - predictions)

    return {
        **metrics,
        "25th_percentile_absolute_error": float(np.quantile(absolute_errors, 0.25)),
        "median_absolute_error": float(np.quantile(absolute_errors, 0.50)),
        "75th_percentile_absolute_error": float(np.quantile(absolute_errors, 0.75)),
        "90th_percentile_absolute_error": float(np.quantile(absolute_errors, 0.90)),
    }


calibration_sample_map = {
    "All complete packages with picks": ("all_complete_packages_with_picks"),
    "Unequal player counts with picks": ("unequal_player_counts_with_picks"),
    "Multi-player packages with picks": ("multi_player_packages_with_picks"),
    "One player versus multiple players with picks": ("one_player_vs_multiple_players_with_picks"),
    "One calculated player for zero with picks": ("one_calculated_player_for_zero_with_picks"),
    "One-for-one calculated players with picks": ("one_for_one_with_picks"),
}

pick_scale_methods = {
    "Prior deployed scale": prior_pick_weights,
    "Unregularized full-development scale": (unregularized_pick_weights),
    "Regularized deployed scale": additive_pick_weights,
}

method_comparison_records = []

for scale_method, method_weights in pick_scale_methods.items():
    for model_name, sample_flag in calibration_sample_map.items():
        sample_data = test_data.loc[test_data[sample_flag]]
        if sample_data.empty:
            continue

        result = evaluate_fixed_pick_weights_with_errors(sample_data, model_name=model_name, pick_weights=method_weights)
        result["scale_method"] = scale_method
        method_comparison_records.append(result)

pick_scale_method_comparison = pd.DataFrame(method_comparison_records)

test_evaluation_results = pick_scale_method_comparison.loc[
    pick_scale_method_comparison["scale_method"].eq("Regularized deployed scale")
].reset_index(drop=True)

display(pick_scale_method_comparison)
display(test_evaluation_results)

,model,row_count,mae,rmse,r_squared,prediction_mean,prediction_standard_deviation,baseline_mae,mae_ratio_to_zero_baseline,target_standard_deviation,25th_percentile_absolute_error,median_absolute_error,75th_percentile_absolute_error,90th_percentile_absolute_error,scale_method
0,All complete packages with picks,144,0.653223,0.831821,-0.830563,0.004231,0.751151,0.502813,1.299138,0.614805,0.350230,0.564312,0.846378,1.196945,Prior deployed scale
1,Unequal player counts with picks,91,0.654507,0.740464,0.006538,0.006695,0.542045,0.665518,0.983455,0.742897,0.389167,0.622103,0.878696,1.030152,Prior deployed scale
2,Multi-player packages with picks,39,0.683570,0.818011,-3.861555,0.009492,0.731762,0.313981,2.177106,0.370998,0.396534,0.622103,1.018217,1.238660,Prior deployed scale
3,One player versus multiple players with picks,24,0.688175,0.835284,-3.976907,0.049812,0.721823,0.288015,2.389376,0.374416,0.399091,0.623200,0.819918,1.394897,Prior deployed scale
4,One calculated player for zero with picks,59,0.631449,0.684509,0.394065,-0.009936,0.398475,0.858356,0.735649,0.879359,0.383575,0.615253,0.844985,0.980092,Prior deployed scale
5,One-for-one calculated players with picks,42,0.581943,0.913886,-9.301462,0.019650,0.977799,0.226590,2.568267,0.284736,0.192348,0.412390,0.588593,1.303895,Prior deployed scale
6,All complete packages with picks,144,0.628042,0.786714,-0.637414,0.003858,0.684947,0.502813,1.249057,0.614805,0.368721,0.540704,0.834824,1.126862,Unregularized full-development scale
7,Unequal player counts with picks,91,0.647731,0.724575,0.048717,0.006105,0.494271,0.665518,0.973273,0.742897,0.404648,0.635051,0.878696,0.964137,Unregularized full-development scale
8,Multi-player packages with picks,39,0.637064,0.761089,-3.208499,0.008655,0.667267,0.313981,2.028989,0.370998,0.370346,0.572631,0.944456,1.155179,Unregularized full-development scale
9,One player versus multiple players with picks,24,0.640050,0.778783,-3.326373,0.045421,0.658204,0.288015,2.222284,0.374416,0.370346,0.579957,0.777769,1.268447,Unregularized full-development scale


,model,row_count,mae,rmse,r_squared,prediction_mean,prediction_standard_deviation,baseline_mae,mae_ratio_to_zero_baseline,target_standard_deviation,25th_percentile_absolute_error,median_absolute_error,75th_percentile_absolute_error,90th_percentile_absolute_error,scale_method
0,All complete packages with picks,144,0.640871,0.809511,-0.733686,0.004050,0.718938,0.502813,1.274572,0.614805,0.353643,0.551946,0.824005,1.164208,Regularized deployed scale
1,Unequal player counts with picks,91,0.651210,0.732387,0.028094,0.006408,0.518800,0.665518,0.978501,0.742897,0.396624,0.625506,0.878696,0.985776,Regularized deployed scale
2,Multi-player packages with picks,39,0.660942,0.790169,-3.536247,0.009085,0.700381,0.313981,2.105037,0.370998,0.381165,0.599158,0.982327,1.198040,Regularized deployed scale
3,One player versus multiple players with picks,24,0.664759,0.807657,-3.653131,0.047675,0.690868,0.288015,2.308074,0.374416,0.381165,0.601596,0.799410,1.333370,Regularized deployed scale
4,One calculated player for zero with picks,59,0.638669,0.689300,0.385554,-0.009510,0.381387,0.858356,0.744061,0.879359,0.399421,0.625506,0.847462,0.969838,Regularized deployed scale
5,One-for-one calculated players with picks,42,0.556290,0.873837,-8.418356,0.018808,0.935867,0.226590,2.455055,0.284736,0.208468,0.397010,0.547573,1.258249,Regularized deployed scale


## Inspect player values, reliability, role capacity, and package concentration

The audits below are designed to catch the exact failure mode that motivated this revision: limited-minute, high-rate role players appearing alongside proven high-minute creators at the top of the value distribution.


In [14]:
# Audit the distribution and reliability of player values.
qualified_player_rows = all_player_value_rows.loc[
    all_player_value_rows["player_value_status"].eq("calculated_efficiency_load_role_adjusted")
].copy()

qualified_player_values = qualified_player_rows["season_player_production_value"].dropna().astype("float64")

player_value_distribution = qualified_player_values.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

reliability_distribution = (
    qualified_player_rows["season_player_reliability"].dropna().astype("float64").describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)

role_capacity_distribution = (
    qualified_player_rows["season_player_role_capacity"].dropna().astype("float64").describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)

display(player_value_distribution.to_frame("qualified_player_production_value"))
display(reliability_distribution.to_frame("player_rate_reliability"))
display(role_capacity_distribution.to_frame("player_role_capacity"))

median_player_value = float(player_value_distribution["50%"])
p90_player_value = float(player_value_distribution["90%"])
p95_player_value = float(player_value_distribution["95%"])

package_examples = {
    "One median player": [median_player_value],
    "Two median players": [median_player_value, median_player_value],
    "Three median players": [median_player_value, median_player_value, median_player_value],
    "One 90th-percentile player": [p90_player_value],
    "One 95th-percentile player": [p95_player_value],
    "90th-percentile player plus median player": [p90_player_value, median_player_value],
    "95th-percentile player plus median player": [p95_player_value, median_player_value],
}

package_sanity_records = []

for package_name, player_values in package_examples.items():
    raw_additive_value = float(np.sum(player_values))
    slot_adjusted_value = calculate_slot_adjusted_package_value(
        player_values, additional_player_slot_cost=(SELECTED_ADDITIONAL_PLAYER_SLOT_COST)
    )

    package_sanity_records.append(
        {
            "comparison": package_name,
            "player_count": len(player_values),
            "raw_additive_value": raw_additive_value,
            "slot_adjusted_value": slot_adjusted_value,
            "slot_cost_deduction": (raw_additive_value - slot_adjusted_value),
        }
    )

package_sanity_table = pd.DataFrame(package_sanity_records)

two_median_adjusted_value = float(
    package_sanity_table.loc[package_sanity_table["comparison"].eq("Two median players"), "slot_adjusted_value"].iloc[0]
)

concentration_summary = pd.DataFrame(
    [
        {
            "comparison": ("95th-percentile player / " "two-median adjusted package"),
            "ratio": (p95_player_value / two_median_adjusted_value if two_median_adjusted_value > 0 else np.inf),
        },
        {
            "comparison": ("90th-percentile player / " "two-median adjusted package"),
            "ratio": (p90_player_value / two_median_adjusted_value if two_median_adjusted_value > 0 else np.inf),
        },
        {
            "comparison": ("Second median player's retained " "contribution share"),
            "ratio": (
                max(median_player_value - SELECTED_ADDITIONAL_PLAYER_SLOT_COST, 0.0) / median_player_value
                if median_player_value > 0
                else np.nan
            ),
        },
    ]
)

reliability_rank = qualified_player_rows["season_player_reliability"].rank(method="first", pct=True)

qualified_player_rows["reliability_quartile"] = pd.cut(
    reliability_rank,
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=["Q1 lowest reliability", "Q2", "Q3", "Q4 highest reliability"],
    include_lowest=True,
)

player_value_reliability_audit = (
    qualified_player_rows.groupby("reliability_quartile", observed=True)
    .agg(
        player_rows=("season_player_production_value", "size"),
        median_minutes=("season_minutes", "median"),
        median_minutes_per_game=("season_minutes_per_game", "median"),
        median_possessions=("season_estimated_player_possessions", "median"),
        median_reliability=("season_player_reliability", "median"),
        median_role_capacity=("season_player_role_capacity", "median"),
        median_raw_quality=("season_player_raw_quality_score", "median"),
        median_role_adjusted_quality=("season_player_role_adjusted_quality_score", "median"),
        median_player_value=("season_player_production_value", "median"),
        p90_player_value=("season_player_production_value", lambda values: values.quantile(0.90)),
        p95_player_value=("season_player_production_value", lambda values: values.quantile(0.95)),
    )
    .reset_index()
)

role_capacity_rank = qualified_player_rows["season_player_role_capacity"].rank(method="first", pct=True)

qualified_player_rows["role_capacity_quartile"] = pd.cut(
    role_capacity_rank,
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=["Q1 lowest role capacity", "Q2", "Q3", "Q4 highest role capacity"],
    include_lowest=True,
)

player_value_role_capacity_audit = (
    qualified_player_rows.groupby("role_capacity_quartile", observed=True)
    .agg(
        player_rows=("season_player_production_value", "size"),
        median_minutes_per_game=("season_minutes_per_game", "median"),
        median_role_capacity=("season_player_role_capacity", "median"),
        median_raw_quality=("season_player_raw_quality_score", "median"),
        median_reliability=("season_player_reliability", "median"),
        median_role_adjusted_quality=("season_player_role_adjusted_quality_score", "median"),
        median_player_value=("season_player_production_value", "median"),
        p90_player_value=("season_player_production_value", lambda values: values.quantile(0.90)),
        p95_player_value=("season_player_production_value", lambda values: values.quantile(0.95)),
    )
    .reset_index()
)

top_player_audit_columns = [
    column
    for column in [
        "transaction_player_name",
        "transaction_date",
        "production_reference_season",
        "season_games_played",
        "season_minutes",
        "season_minutes_per_game",
        "season_estimated_player_possessions",
        "season_player_reliability",
        "season_player_role_capacity",
        "season_true_shooting_attempts_per_100",
        "season_true_shooting_percentage",
        "season_efficiency_points_added_per_100",
        "season_assists_per_100",
        "season_turnovers_per_100",
        "scoring_load_quality_score",
        "scoring_efficiency_quality_score",
        "playmaking_quality_score",
        "rebounding_defensive_events_quality_score",
        "overall_impact_quality_score",
        "season_player_raw_quality_score",
        "season_player_shrunk_quality_score",
        "season_player_role_adjusted_quality_score",
        "season_role_adjusted_base_value",
        "season_quality_component",
        "season_player_production_value",
    ]
    if column in qualified_player_rows.columns
]

top_player_value_audit = (
    qualified_player_rows[top_player_audit_columns]
    .sort_values("season_player_production_value", ascending=False)
    .drop_duplicates(subset=["transaction_player_name", "transaction_date"])
    .head(75)
    .reset_index(drop=True)
)

# Explicitly inspect high-rate observations with less than 18 MPG.
limited_role_high_value_audit = (
    qualified_player_rows.loc[qualified_player_rows["season_minutes_per_game"].lt(18.0), top_player_audit_columns]
    .sort_values("season_player_production_value", ascending=False)
    .drop_duplicates(subset=["transaction_player_name", "transaction_date"])
    .head(50)
    .reset_index(drop=True)
)

display(package_sanity_table)
display(concentration_summary)
display(player_value_reliability_audit)
display(player_value_role_capacity_audit)
display(top_player_value_audit)
display(limited_role_high_value_audit)

,qualified_player_production_value
count,4449.000000
mean,0.872910
std,0.245553
min,0.374012
10%,0.572231
25%,0.684293
50%,0.850041
75%,1.040904
90%,1.202426
95%,1.316346


,player_rate_reliability
count,4449.000000
mean,0.835700
std,0.121943
min,0.434055
10%,0.641418
25%,0.782959
50%,0.881446
75%,0.926295
90%,0.947171
max,0.962950


,player_role_capacity
count,4449.000000
mean,0.808304
std,0.163979
min,0.347750
10%,0.571922
25%,0.676695
50%,0.830312
75%,0.974363
90%,1.000000
max,1.000000


,comparison,player_count,raw_additive_value,slot_adjusted_value,slot_cost_deduction
0,One median player,1,0.850041,0.850041,0.0
1,Two median players,2,1.700083,1.200083,0.5
2,Three median players,3,2.550124,1.550124,1.0
3,One 90th-percentile player,1,1.202426,1.202426,0.0
4,One 95th-percentile player,1,1.316346,1.316346,0.0
5,90th-percentile player plus median player,2,2.052468,1.552468,0.5
6,95th-percentile player plus median player,2,2.166388,1.666388,0.5


,comparison,ratio
0,95th-percentile player / two-median adjusted p...,1.096880
1,90th-percentile player / two-median adjusted p...,1.001953
2,Second median player's retained contribution s...,0.411793


,reliability_quartile,player_rows,median_minutes,median_minutes_per_game,median_possessions,median_reliability,median_role_capacity,median_raw_quality,median_role_adjusted_quality,median_player_value,p90_player_value,p95_player_value
0,Q1 lowest reliability,1112,260.0,10.918919,509.593961,0.670877,0.603294,-0.219607,-0.092679,0.631519,0.880244,0.972373
1,Q2,1112,681.0,16.500000,1347.061113,0.843462,0.741620,-0.097054,-0.061713,0.760737,1.022899,1.135105
2,Q3,1112,1237.0,22.440556,2419.980938,0.906366,0.864881,0.050048,0.040937,0.907003,1.163349,1.261476
3,Q4 highest reliability,1113,2152.0,31.040000,4228.409426,0.944177,1.000000,0.271608,0.254540,1.100519,1.368655,1.450942


,role_capacity_quartile,player_rows,median_minutes_per_game,median_role_capacity,median_raw_quality,median_reliability,median_role_adjusted_quality,median_player_value,p90_player_value,p95_player_value
0,Q1 lowest role capacity,1112,10.509091,0.591864,-0.261707,0.697517,-0.106407,0.611588,0.742349,0.772970
1,Q2,1112,17.298701,0.759357,-0.132362,0.865797,-0.079657,0.764699,0.930243,0.975321
2,Q3,1112,24.258065,0.899223,0.046924,0.908389,0.036748,0.939018,1.125133,1.198571
3,Q4 highest role capacity,1113,33.140000,1.000000,0.303457,0.940770,0.278306,1.138833,1.392661,1.458166


,transaction_player_name,transaction_date,production_reference_season,season_games_played,season_minutes,season_minutes_per_game,season_estimated_player_possessions,season_player_reliability,season_player_role_capacity,season_true_shooting_attempts_per_100,...,scoring_efficiency_quality_score,playmaking_quality_score,rebounding_defensive_events_quality_score,overall_impact_quality_score,season_player_raw_quality_score,season_player_shrunk_quality_score,season_player_role_adjusted_quality_score,season_role_adjusted_base_value,season_quality_component,season_player_production_value
0,LeBron James,2010-07-09,2009-10,76.0,2929.0,38.539474,5596.599124,0.957240,1.00000,33.379557,...,2.838849,1.186192,0.517116,2.395022,2.175283,2.082268,2.082268,1.000000,1.041134,2.041134
1,Isaiah Thomas,2017-08-22,2016-17,76.0,2529.0,33.276316,5125.613507,0.953494,1.00000,34.309259,...,3.000000,0.708608,-0.768899,1.309542,1.796693,1.713136,1.713136,1.000000,0.856568,1.856568
2,Chris Paul,2017-06-28,2016-17,61.0,1891.0,31.000000,3793.032110,0.938165,1.00000,23.711901,...,2.313628,1.787411,0.404986,3.000000,1.697595,1.592624,1.592624,1.000000,0.796312,1.796312
3,Kyrie Irving,2017-08-22,2016-17,72.0,2488.0,34.555556,5026.800513,0.952623,1.00000,31.119596,...,1.951059,0.770513,-0.505387,1.531741,1.525368,1.453101,1.453101,1.000000,0.726550,1.726550
4,Kevin Love,2014-08-23,2013-14,77.0,2756.0,35.792208,5616.461173,0.957385,1.00000,30.259623,...,2.222022,0.293570,0.281887,1.478609,1.477009,1.414066,1.414066,1.000000,0.707033,1.707033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Tobias Harris,2019-02-06,2018-19,55.0,1877.0,34.127273,3970.828277,0.940770,1.00000,23.958729,...,2.143773,0.004812,-0.238137,0.443195,0.785661,0.739126,0.739126,1.000000,0.369563,1.369563
71,Carmelo Anthony,2017-09-25,2016-17,74.0,2501.0,33.797297,5039.882952,0.952740,1.00000,30.746746,...,0.597164,0.017041,-0.334777,-0.425483,0.766255,0.730041,0.730041,1.000000,0.365021,1.365021
72,Kenny Anderson,1996-01-19,1995-96,31.0,1042.0,33.612903,1965.700690,0.887169,1.00000,23.539698,...,-0.022919,1.918703,-0.049758,0.377577,0.813442,0.721661,0.721661,1.000000,0.360830,1.360830
73,Jeff Teague,2016-07-07,2015-16,79.0,2213.0,28.012658,4506.220102,0.947437,0.96631,24.971705,...,0.904314,0.862681,-0.233242,0.635848,0.845319,0.800887,0.773905,0.973048,0.386952,1.360001


,transaction_player_name,transaction_date,production_reference_season,season_games_played,season_minutes,season_minutes_per_game,season_estimated_player_possessions,season_player_reliability,season_player_role_capacity,season_true_shooting_attempts_per_100,...,scoring_efficiency_quality_score,playmaking_quality_score,rebounding_defensive_events_quality_score,overall_impact_quality_score,season_player_raw_quality_score,season_player_shrunk_quality_score,season_player_role_adjusted_quality_score,season_role_adjusted_base_value,season_quality_component,season_player_production_value
0,Montrezl Harrell,2017-06-28,2016-17,58.0,1032.0,17.793103,2167.383498,0.896582,0.770132,18.374229,...,2.703468,0.037421,0.180251,1.235129,0.715234,0.641266,0.493860,0.816106,0.246930,1.063035
1,Michael Beasley,2016-09-22,2015-16,20.0,353.0,17.650000,720.109353,0.742297,0.767029,31.461888,...,1.434827,-0.524412,0.474887,-1.238600,0.802390,0.595612,0.456852,0.813623,0.228426,1.042049
2,Cedric Ceballos,1998-02-18,1997-98,34.0,610.0,17.941176,1181.264096,0.825329,0.773330,24.969861,...,1.239111,-0.348193,0.292170,0.701087,0.671771,0.554432,0.428759,0.818664,0.214380,1.033044
3,Devin Harris,2018-02-08,2017-18,44.0,784.0,17.818182,1565.882758,0.862326,0.770675,21.283841,...,0.998387,0.258883,-0.171771,1.653285,0.640058,0.551938,0.425365,0.816540,0.212682,1.029222
4,Leandro Barbosa,2010-07-14,2009-10,44.0,765.0,17.386364,1554.844562,0.861484,0.761279,25.540817,...,0.323403,0.015851,-0.522796,1.416560,0.641366,0.552526,0.420627,0.809023,0.210313,1.019336
5,Jerryd Bayless,2010-10-23,2009-10,74.0,1264.0,17.081081,2335.752694,0.903316,0.754566,25.237261,...,0.518935,0.421144,-0.810348,0.332956,0.571412,0.516166,0.389481,0.803653,0.194741,0.998393
6,Brandan Wright,2015-01-09,2014-15,35.0,574.0,16.400000,1135.381616,0.819544,0.739369,15.755055,...,3.000000,-0.134489,1.157054,1.249635,0.666728,0.546413,0.404001,0.791495,0.202001,0.993496
7,Shawn Long,2017-06-28,2016-17,17.0,226.0,13.294118,467.969750,0.651796,0.665686,26.582915,...,2.076562,-0.230067,1.301503,1.119955,1.133983,0.739126,0.492025,0.732548,0.246013,0.978561
8,Marcus Thornton,2015-02-19,2014-15,39.0,621.0,15.923077,1241.763011,0.832413,0.728539,26.368961,...,0.453988,0.067938,-0.366786,0.643360,0.632199,0.526250,0.383394,0.782831,0.191697,0.974528
9,Boris Diaw,2016-07-07,2015-16,76.0,1348.0,17.736842,2657.742756,0.914023,0.768914,15.668936,...,1.152815,0.368638,-0.381200,2.279499,0.418312,0.382346,0.293991,0.815131,0.146996,0.962126


## Inspect the deployed pick scale against the finalized player-value distribution

These diagnostics compare numeric scales only. They do not claim that a particular historical player-season is a market equivalent for a pick.


In [15]:
# Summarize qualified player values for package comparisons.
qualified_player_values = (
    all_player_value_rows.loc[
        all_player_value_rows["player_value_status"].eq("calculated_efficiency_load_role_adjusted"), "season_player_production_value"
    ]
    .dropna()
    .astype("float64")
)

if qualified_player_values.empty:
    raise ValueError("No qualified player values are available for pick-scale diagnostics.")

player_value_reference_points = {
    "median_player_value": float(qualified_player_values.quantile(0.50)),
    "p75_player_value": float(qualified_player_values.quantile(0.75)),
    "p90_player_value": float(qualified_player_values.quantile(0.90)),
    "p95_player_value": float(qualified_player_values.quantile(0.95)),
    "p99_player_value": float(qualified_player_values.quantile(0.99)),
}

pick_scale_distribution_audit = pd.DataFrame(
    [
        {
            "tier": row.tier,
            "relative_curve_weight": (row.relative_curve_weight),
            "prior_value": row.prior_estimated_value,
            "unregularized_value": (row.unregularized_estimated_value),
            "regularized_value": row.estimated_value,
            "percentage_change_from_prior": (row.percentage_change_from_prior),
            "regularized_player_value_percentile": (percentileofscore(qualified_player_values, row.estimated_value, kind="weak")),
            "regularized_value_to_median_player": (row.estimated_value / player_value_reference_points["median_player_value"]),
            "regularized_value_to_p90_player": (row.estimated_value / player_value_reference_points["p90_player_value"]),
            "regularized_value_to_p95_player": (row.estimated_value / player_value_reference_points["p95_player_value"]),
        }
        for row in additive_pick_weight_table.itertuples(index=False)
    ]
)

player_value_reference_table = pd.DataFrame([player_value_reference_points])

display(player_value_reference_table)
display(pick_scale_distribution_audit)

,median_player_value,p75_player_value,p90_player_value,p95_player_value,p99_player_value
0,0.850041,1.040904,1.202426,1.316346,1.497594


,tier,relative_curve_weight,prior_value,unregularized_value,regularized_value,percentage_change_from_prior,regularized_player_value_percentile,regularized_value_to_median_player,regularized_value_to_p90_player,regularized_value_to_p95_player
0,projected_late_second,0.168931,0.239096,0.218023,0.228842,-4.288412,0.000000,0.269213,0.190317,0.173847
1,projected_early_second,0.168931,0.239096,0.218023,0.228842,-4.288412,0.000000,0.269213,0.190317,0.173847
2,projected_late_first,0.414182,0.586212,0.534546,0.561073,-4.288412,8.361430,0.660054,0.466617,0.426235
3,projected_lottery_first,0.844753,1.195620,1.090243,1.144347,-4.288412,85.592268,1.346225,0.951698,0.869336
4,projected_top_5_first,1.000000,1.415349,1.290606,1.354653,-4.288412,96.538548,1.593632,1.126600,1.029101


## Save finalized player values and regularized pick-calibration outputs


In [16]:
# Save the final model bundle, pick values, and audit artifacts.
metric_reference_path = (
    processed_data_path / "player_production_value_metric_reference_efficiency_load_role_scaled_base_regularized.parquet"
)
player_values_path = processed_data_path / "transaction_player_production_values_efficiency_load_role_scaled_base_regularized.parquet"
scored_model_path = processed_data_path / "draft_weight_model_data_efficiency_load_role_scaled_base_regularized_scored.parquet"
calibration_results_path = processed_data_path / "draft_weight_efficiency_load_role_scaled_base_regularized_test_results.parquet"
hyperparameter_search_path = processed_data_path / "efficiency_load_role_scaled_base_hyperparameter_search_regularized.parquet"
regularization_search_path = processed_data_path / "pick_scale_regularization_search_efficiency_load_role_scaled_base.parquet"
pick_scale_method_comparison_path = (
    processed_data_path / "pick_scale_method_comparison_efficiency_load_role_scaled_base_regularized.parquet"
)
pick_scale_bootstrap_path = processed_data_path / "pick_scale_bootstrap_efficiency_load_role_scaled_base_regularized.parquet"
pick_values_path = processed_data_path / "anchored_efficiency_load_role_scaled_base_regularized_pick_values.parquet"
reliability_audit_path = processed_data_path / "player_value_reliability_audit_efficiency_load_role_scaled_base_regularized.parquet"
role_capacity_audit_path = processed_data_path / "player_value_role_capacity_audit_efficiency_load_role_scaled_base_regularized.parquet"
top_player_audit_path = processed_data_path / "top_player_value_audit_efficiency_load_role_scaled_base_regularized.parquet"
limited_role_audit_path = processed_data_path / "limited_role_high_value_audit_efficiency_load_role_scaled_base_regularized.parquet"
model_bundle_path = processed_data_path / "draft_weight_model_bundle_efficiency_load_role_scaled_base_regularized.json"

final_metric_reference.to_parquet(metric_reference_path, index=False)
all_player_value_rows.to_parquet(player_values_path, index=False)
scored_model_data.to_parquet(scored_model_path, index=False)
test_evaluation_results.to_parquet(calibration_results_path, index=False)
hyperparameter_search.to_parquet(hyperparameter_search_path, index=False)
regularization_search.to_parquet(regularization_search_path, index=False)
pick_scale_method_comparison.to_parquet(pick_scale_method_comparison_path, index=False)
pick_scale_bootstrap.to_parquet(pick_scale_bootstrap_path, index=False)
additive_pick_weight_table.to_parquet(pick_values_path, index=False)
player_value_reliability_audit.to_parquet(reliability_audit_path, index=False)
player_value_role_capacity_audit.to_parquet(role_capacity_audit_path, index=False)
top_player_value_audit.to_parquet(top_player_audit_path, index=False)
limited_role_high_value_audit.to_parquet(limited_role_audit_path, index=False)

player_metric_reference_dict = {
    row.metric: {"mean": float(row.mean), "std_ddof_0": float(row.std_ddof_0), "reference_count": int(row.reference_count)}
    for row in final_metric_reference.itertuples(index=False)
}

reference_true_shooting_percentage = float(final_metric_reference["reference_true_shooting_percentage"].dropna().iloc[0])


def dataframe_records_json_safe(dataframe):
    records = []
    for record in dataframe.to_dict(orient="records"):
        cleaned = {}
        for key, value in record.items():
            if isinstance(value, (np.floating, np.integer)):
                cleaned[key] = value.item()
            elif isinstance(value, (np.bool_,)):
                cleaned[key] = bool(value)
            elif isinstance(value, pd.Timestamp):
                cleaned[key] = value.isoformat()
            elif pd.isna(value):
                cleaned[key] = None
            else:
                cleaned[key] = value
        records.append(cleaned)
    return records


model_bundle = {
    "model_type": ("role_scaled_base_player_production_with_fixed_relative_" "pick_curve_and_regularized_absolute_scale"),
    "player_value_column": "season_player_production_value",
    "player_quality_column": "season_player_quality_score",
    "raw_player_quality_column": "season_player_raw_quality_score",
    "reliability_shrunk_quality_column": ("season_player_shrunk_quality_score"),
    "role_adjusted_quality_column": ("season_player_role_adjusted_quality_score"),
    "player_reliability_column": "season_player_reliability",
    "player_role_capacity_column": "season_player_role_capacity",
    "quality_component_column": "season_quality_component",
    "minimum_season_minutes": MINIMUM_SEASON_MINUTES,
    "base_player_value": BASE_PLAYER_VALUE,
    "minimum_player_base": MINIMUM_PLAYER_BASE,
    "role_adjusted_base_value_column": ("season_role_adjusted_base_value"),
    "quality_multiplier_slope": QUALITY_MULTIPLIER_SLOPE,
    "zscore_clip_limit": ZSCORE_CLIP_LIMIT,
    "reference_true_shooting_percentage": (reference_true_shooting_percentage),
    "shooting_load_formula": (
        "100 * (season_field_goals_attempted + " "0.44 * season_free_throws_attempted) / " "season_estimated_player_possessions"
    ),
    "efficiency_points_added_formula": (
        "2 * season_true_shooting_attempts_per_100 * " "(season_true_shooting_percentage - " "reference_true_shooting_percentage)"
    ),
    "turnovers_per_100_formula": ("100 * season_turnovers / " "season_estimated_player_possessions"),
    "playmaking_formula": ("(2/3) * assists_per_100_zscore - " "(1/3) * turnovers_per_100_zscore"),
    "game_score_policy": ("Game Score is excluded to avoid duplicating scoring and " "other box-score production."),
    "reliability_shrinkage_constant": (SELECTED_RELIABILITY_SHRINKAGE_CONSTANT),
    "reliability_formula": (
        "season_estimated_player_possessions / " "(season_estimated_player_possessions + " "reliability_shrinkage_constant)"
    ),
    "role_capacity_full_minutes_per_game": (ROLE_CAPACITY_FULL_MINUTES_PER_GAME),
    "role_capacity_exponent": ROLE_CAPACITY_EXPONENT,
    "role_capacity_formula": ("clip(season_minutes_per_game / " "role_capacity_full_minutes_per_game, 0, 1) " "** role_capacity_exponent"),
    "individual_player_value_floor": 0.0,
    "individual_player_value_formula": (
        "if season_minutes >= minimum_season_minutes: "
        "raw_quality = weighted sum of clipped component scores; "
        "reliability = possessions / "
        "(possessions + reliability_shrinkage_constant); "
        "role_capacity = clip(minutes_per_game / 30, 0, 1) ** 0.5; "
        "role_adjusted_quality = reliability * role_capacity * raw_quality; "
        "role_scaled_base = minimum_player_base + "
        "(base_player_value - minimum_player_base) * role_capacity; "
        "value = max(0, role_scaled_base + "
        "quality_multiplier_slope * role_adjusted_quality); "
        "below-threshold players are unknown; "
        "only no-prior-NBA-production players are known zero"
    ),
    "additional_player_slot_cost": (SELECTED_ADDITIONAL_PLAYER_SLOT_COST),
    "package_value_formula": (
        "sort individual player values descending; "
        "the highest-valued player contributes full value; "
        "each additional player contributes "
        "max(individual_value - additional_player_slot_cost, 0)"
    ),
    "player_hyperparameter_selection": {
        "candidate_reliability_shrinkage_constants": (CANDIDATE_RELIABILITY_SHRINKAGE_CONSTANTS),
        "candidate_additional_player_slot_costs": (CANDIDATE_ADDITIONAL_PLAYER_SLOT_COSTS),
        "parameter_training_seasons": parameter_training_seasons,
        "validation_seasons": validation_seasons,
        "validation_sample_weights": VALIDATION_SAMPLE_WEIGHTS,
        "validation_score_tolerance": VALIDATION_SCORE_TOLERANCE,
        "selected_row": dataframe_records_json_safe(pd.DataFrame([selected_hyperparameter_row]))[0],
    },
    "player_metric_columns": player_metric_columns,
    "player_metric_reference": player_metric_reference_dict,
    "player_domains": player_domains,
    "player_domain_metric_weights": {
        domain_name: {metric_name: float(metric_weight) for metric_name, metric_weight in metric_weights.items()}
        for domain_name, metric_weights in (player_domain_metric_weights.items())
    },
    "player_domain_weights": {key: float(value) for key, value in player_domain_weights.items()},
    "direct_metric_weights": {key: float(value) for key, value in direct_metric_weights.items()},
    "current_roster_metric_mapping": current_roster_metric_mapping,
    "current_roster_raw_column_mapping": (current_roster_raw_column_mapping),
    "original_quality_model_weights": {key: float(value) for key, value in original_quality_model_weights.items()},
    "relative_pick_curve": {tier: float(value) for tier, value in relative_pick_curve.items()},
    "pick_scale_calibration": {
        "primary_sample": PRIMARY_PICK_SCALE_SAMPLE_FLAG,
        "prior_deployed_pick_values": {key: float(value) for key, value in PRIOR_DEPLOYED_PICK_VALUES.items()},
        "prior_pick_curve_scale": PRIOR_PICK_CURVE_SCALE,
        "candidate_regularization_strengths": (CANDIDATE_PICK_SCALE_REGULARIZATION_STRENGTHS),
        "validation_sample_weights": (PICK_SCALE_VALIDATION_SAMPLE_WEIGHTS),
        "validation_score_tolerance": (PICK_SCALE_VALIDATION_SCORE_TOLERANCE),
        "selected_regularization_strength": (SELECTED_PICK_SCALE_REGULARIZATION_STRENGTH),
        "unregularized_full_development_scale": (unregularized_pick_curve_scale),
        "selected_regularized_scale": selected_pick_curve_scale,
        "regularization_objective": (
            "normalized_training_mae + regularization_strength * " "((candidate_scale - prior_scale) / prior_scale) ** 2"
        ),
        "selection_summary": dataframe_records_json_safe(regularization_selection_summary),
        "full_development_fit": {
            key: (value.item() if isinstance(value, (np.floating, np.integer)) else value)
            for key, value in (full_development_regularized_fit.items())
            if key != "weights"
        },
        "bootstrap_summary": dataframe_records_json_safe(bootstrap_scale_summary)[0],
    },
    "selected_pick_curve_scale": selected_pick_curve_scale,
    "outright_pick_hierarchy": outright_pick_hierarchy,
    "outright_pick_net_columns": outright_pick_net_columns,
    "outright_pick_weights": additive_pick_weights.tolist(),
    "intercept": 0.0,
    "balance_equation": (
        "slot-adjusted player package value received - "
        "slot-adjusted player package value sent + "
        "pick value received - pick value sent"
    ),
    "swap_policy": ("Swaps are excluded from calibration and application predictions."),
    "sign_convention": (
        "Positive player-value differential means more slot-adjusted "
        "player value acquired. Positive net draft count means draft "
        "compensation received."
    ),
    "development_seasons": development_seasons,
    "test_seasons": test_seasons,
    "regularization_search": dataframe_records_json_safe(regularization_search),
    "pick_scale_method_comparison": dataframe_records_json_safe(pick_scale_method_comparison),
    "test_evaluation_results": dataframe_records_json_safe(test_evaluation_results),
    "player_value_distribution": {key: float(value) for key, value in player_value_distribution.items()},
    "reliability_distribution": {key: float(value) for key, value in reliability_distribution.items()},
    "role_capacity_distribution": {key: float(value) for key, value in role_capacity_distribution.items()},
    "package_sanity_table": dataframe_records_json_safe(package_sanity_table),
    "player_value_reliability_audit": dataframe_records_json_safe(player_value_reliability_audit),
    "player_value_role_capacity_audit": dataframe_records_json_safe(player_value_role_capacity_audit),
    "pick_scale_distribution_audit": dataframe_records_json_safe(pick_scale_distribution_audit),
}

with open(model_bundle_path, "w", encoding="utf-8") as file:
    json.dump(model_bundle, file, indent=2)

print("Saved:", metric_reference_path)
print("Saved:", player_values_path)
print("Saved:", scored_model_path)
print("Saved:", calibration_results_path)
print("Saved:", hyperparameter_search_path)
print("Saved:", regularization_search_path)
print("Saved:", pick_scale_method_comparison_path)
print("Saved:", pick_scale_bootstrap_path)
print("Saved:", pick_values_path)
print("Saved:", reliability_audit_path)
print("Saved:", role_capacity_audit_path)
print("Saved:", top_player_audit_path)
print("Saved:", limited_role_audit_path)
print("Saved:", model_bundle_path)

Saved: ..\data\processed\player_production_value_metric_reference_efficiency_load_role_scaled_base_regularized.parquet
Saved: ..\data\processed\transaction_player_production_values_efficiency_load_role_scaled_base_regularized.parquet
Saved: ..\data\processed\draft_weight_model_data_efficiency_load_role_scaled_base_regularized_scored.parquet
Saved: ..\data\processed\draft_weight_efficiency_load_role_scaled_base_regularized_test_results.parquet
Saved: ..\data\processed\efficiency_load_role_scaled_base_hyperparameter_search_regularized.parquet
Saved: ..\data\processed\pick_scale_regularization_search_efficiency_load_role_scaled_base.parquet
Saved: ..\data\processed\pick_scale_method_comparison_efficiency_load_role_scaled_base_regularized.parquet
Saved: ..\data\processed\pick_scale_bootstrap_efficiency_load_role_scaled_base_regularized.parquet
Saved: ..\data\processed\anchored_efficiency_load_role_scaled_base_regularized_pick_values.parquet
Saved: ..\data\processed\player_value_reliability